# TABLES

Loads finished results and writes them as LaTeX tables for the thesis. Nothing is trained here.

One section per table. Every table is shown as a DataFrame and written to `tables/<name>.tex`
(booktabs; add `\usepackage{booktabs,multirow}` and `\usepackage[flushleft]{threeparttable}` to the preamble).

**The baseline is MSP.** It uses the `threshold` variant, as in `C2_evaluation.ipynb`: the probability of the class C0
predicted at its Youden threshold. It is computed per fold on the same rows as the models.

`SHOW_MSP` in the setup cell turns the MSP column on or off in every table. When it is off, bold marks the best model in each row.

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

sys.path.insert(0, os.path.abspath("code"))
from c2_msp import msp_score

RESULTS = os.environ.get("THESIS_RESULTS", "/work3/s251710/thesis_results")
C2      = os.path.join(RESULTS, "C2_final_results")
POLICY  = "ignore"
MSP_VARIANT = "threshold"
# MSP column in every table. Off -> the column, its rule and its bold all disappear, and
# bold then marks the best model in each row.
SHOW_MSP  = False
MSP_LABEL = "Adjusted MSP"
N_FOLDS = 5

DISEASES = ["atelectasis", "cardiomegaly", "consolidation", "edema", "effusion"]
DISEASE_NAMES = {d: d.capitalize() for d in DISEASES}

TABLE_DIR = os.path.abspath("tables")
os.makedirs(TABLE_DIR, exist_ok=True)

In [2]:
def c0_threshold(disease, policy=POLICY):
    p = os.path.join(RESULTS, "C0_final", f"multilabel_{policy}", disease, "threshold.txt")
    return float(open(p).read().strip())

def fold_predictions(run_dir, n_folds=N_FOLDS):
    """Per-fold prediction frames of one run, in fold order."""
    return [pd.read_csv(os.path.join(run_dir, f"fold_{i}_predictions.csv")) for i in range(n_folds)]

def msp_fold_aucs(run_dir, disease, variant=MSP_VARIANT):
    """MSP AUROC for correctness on each fold's test rows -- untrained, straight off C0."""
    t = None if variant == "standard" else c0_threshold(disease)
    return np.array([roc_auc_score(f["correct"], msp_score(f[f"{disease}_prob"].values, t, variant))
                     for f in fold_predictions(run_dir)])

def summary_fold_aucs(run_dir):
    """cv_summary.csv -> {config: array of per-fold AUROCs}."""
    s = pd.read_csv(os.path.join(run_dir, "cv_summary.csv"))
    return {r.config: np.array([float(x) for x in r.auc_folds.split(";")]) for r in s.itertuples()}

In [ ]:
def fmt_cell(folds, bold=False, digits=3, sig=""):
    """mean +- std across folds, e.g. $0.858 \\pm 0.005$. `sig`: stars, printed as a superscript on the mean."""
    m, s = folds.mean(), folds.std()
    sup = f"^{{{sig}}}" if sig else ""
    mean = f"\\mathbf{{{m:.{digits}f}}}" if bold else f"{m:.{digits}f}"
    return f"${mean}{sup} \\pm {s:.{digits}f}$"

def latex_table(rows, columns, group_col, caption, label, col_fmt=None, header=None, notes=None,
                merge_cols=(), resize=False, pre_header=(), group_rules=True):
    """Booktabs table from a list of dicts. `group_col` is multirow-merged and a midrule is
    drawn between groups. `merge_cols` are constant within a group and printed once, as a
    multirow taken from the group's first row. `resize` scales the tabular to \\textwidth
    (needs graphicx). `notes` go in a threeparttable, left-aligned and as wide as the tabular. `pre_header` are raw LaTeX lines placed above the column names. `group_rules=False`
    drops the midrule between groups, for tables with one row per group. Cells are pre-formatted strings."""
    assert not (notes and resize), "threeparttable cannot measure a resized tabular"
    col_fmt = col_fmt or "l" * len(columns)
    header  = header or columns
    groups  = pd.Series([r[group_col] for r in rows])
    lines = ["\\begin{table}[htbp]", "\\centering", "\\small",
             f"\\caption{{{caption}}}", f"\\label{{{label}}}",
             *(["\\begin{threeparttable}"] if notes else []),
             *(["\\resizebox{\\textwidth}{!}{%"] if resize else []),
             f"\\begin{{tabular}}{{{col_fmt}}}", "\\toprule", *pre_header,
             " & ".join(header) + " \\\\", "\\midrule"]
    prev = None
    for i, r in enumerate(rows):
        g = r[group_col]
        if group_rules and prev is not None and g != prev:
            lines.append("\\midrule")
        n = int((groups == g).sum())
        first = g != prev
        cells = [(f"\\multirow{{{n}}}{{*}}{{{g}}}" if n > 1 else g) if first else ""]
        for c in columns:
            if c == group_col:
                continue
            if c in merge_cols:
                cells.append((f"\\multirow{{{n}}}{{*}}{{{r[c]}}}" if n > 1 else str(r[c])) if first else "")
            else:
                cells.append(str(r[c]))
        lines.append(" & ".join(cells) + " \\\\")
        prev = g
    lines += ["\\bottomrule", "\\end{tabular}" + ("}" if resize else "")]
    if notes:
        lines += ["\\begin{tablenotes}\\footnotesize", f"\\item {notes}", "\\end{tablenotes}",
                  "\\end{threeparttable}"]
    lines.append("\\end{table}")
    return "\n".join(lines)

def msp_cols():
    """["MSP"] when the MSP column is on, else [] -- splice into column / key lists."""
    return ["MSP"] if SHOW_MSP else []

def msp_col_fmt(n_models):
    """Disease, K, [MSP |] models."""
    return "ll" + ("c|" if SHOW_MSP else "") + "c" * n_models

def save_table(tex, name):
    path = os.path.join(TABLE_DIR, f"{name}.tex")
    with open(path, "w") as fh:
        fh.write(tex + "\n")
    print(f"wrote {path}\n")
    print(tex)

## 0. C0: the data split and the task it hands to C2

The data is split once, at patient level, into two disjoint halves: the **C0 half** (C0 training + checkpoint selection)
and the **held-out half**, which C0 never sees and which is C2's training and test data (5-fold CV).

* **Table A (`c0_split`): the split.** Per task: the full set before the split (the two halves added up), then per half: images with a confident label (U-Ignore masks `-1` per
  label, so counts differ by task), the positive rate, and C0's AUROC. On the C0 half the AUROC is on the selection set,
  because C0 was never run on its own training images. The mean row is the usual mean competition AUROC.
* **Table B (`c2_data`): the correctness task C0 creates.** Per task: the held-out rows, the Youden threshold (fit on the
  selection set), and C0's errors at that threshold (the `correct = 0` class C2 learns to detect), split into FP / FN.

A task's held-out rows are its confident rows in `C2_dataset.csv` minus the 8 to 10 images tagged with a lateral view
(`AP/PA` = `LL`/`RL`), which the C2 pipeline drops. `correct` is checked against `(prob >= threshold) == label` below.

In [4]:
C0_DIR   = os.path.join(RESULTS, "C0_final", f"multilabel_{POLICY}")
C0_LABEL = {"atelectasis": "Atelectasis", "cardiomegaly": "Cardiomegaly", "consolidation": "Consolidation",
            "edema": "Edema", "effusion": "Pleural Effusion"}
C0_SPLITS = {s: pd.read_csv(f"{s}.csv") for s in ["C0_train", "C0_checkpoint_selection", "C2_dataset"]}

def c0_preds(disease, split):
    """C0's probability and label for one task on one split; uncertain rows (true = NaN) dropped."""
    p = pd.read_csv(os.path.join(C0_DIR, disease, f"{split}_c0_{disease}.csv"), usecols=["path", "prob", "true"])
    return p.dropna(subset=["true"])

def c0_tables():
    """Returns (Table A frame, Table B frame), one row per disease, numeric."""
    A, B = [], []
    for d in DISEASES:
        # C0 half: confident labels of C0_train + C0_checkpoint_selection.
        lab = pd.concat([C0_SPLITS[s][C0_LABEL[d]] for s in ["C0_train", "C0_checkpoint_selection"]])
        c0_half = lab[lab.isin([0, 1])]
        sel = c0_preds(d, "C0_checkpoint_selection")

        # Held-out half = the union of the C2 fold test sets; labels from C2_dataset.csv.
        # Any C2 run holds the same rows; this is the attribute run of Section 1 at K=10.
        oof = pd.concat(fold_predictions(os.path.join(C2, d, "knn_correct_cf", "cf_10", "attributes")))
        y = C0_SPLITS["C2_dataset"].set_index("Path").loc[oof["path"], C0_LABEL[d]].values
        assert set(np.unique(y)) <= {0, 1}, d
        thr = c0_threshold(d)
        pred = (oof[f"{d}_prob"].values >= thr).astype(int)
        assert ((pred == y) == oof["correct"].values.astype(bool)).all(), f"{d}: `correct` != (prob >= thr) == label"

        A.append({"disease": d, "all_n": len(c0_half) + len(oof),
                  "all_pos": (c0_half.sum() + y.sum()) / (len(c0_half) + len(oof)),
                  "c0_n": len(c0_half), "c0_pos": c0_half.mean(),
                  "c0_auc": roc_auc_score(sel["true"], sel["prob"]),
                  "c2_n": len(oof), "c2_pos": y.mean(), "c2_auc": roc_auc_score(y, oof[f"{d}_prob"])})
        B.append({"disease": d, "n": len(oof), "pos": y.mean(), "threshold": thr,
                  "errors": int((pred != y).sum()), "err_rate": (pred != y).mean(),
                  "fp": int(((pred == 1) & (y == 0)).sum()), "fn": int(((pred == 0) & (y == 1)).sum())})
    return pd.DataFrame(A).set_index("disease"), pd.DataFrame(B).set_index("disease")

c0_A, c0_B = c0_tables()
display(c0_A.round(4))
c0_B.round(4)

,all_n,all_pos,c0_n,c0_pos,c0_auc,c2_n,c2_pos,c2_auc
disease,,,,,,,,
atelectasis,161156,0.1844,80218,0.1824,0.7117,80938,0.1864,0.7151
cardiomegaly,184312,0.1269,91805,0.1288,0.8694,92507,0.1250,0.8670
consolidation,166637,0.0779,82949,0.0777,0.7442,83688,0.0781,0.7358
edema,179199,0.2772,89347,0.2767,0.8511,89852,0.2777,0.8469
effusion,181440,0.4238,90506,0.4255,0.8819,90934,0.4222,0.8842


,n,pos,threshold,errors,err_rate,fp,fn
disease,,,,,,,
atelectasis,80938,0.1864,0.1886,32809,0.4054,29028,3781
cardiomegaly,92507,0.1250,0.1598,17548,0.1897,14769,2779
consolidation,83688,0.0781,0.0671,32530,0.3887,30845,1685
edema,89852,0.2777,0.2849,22615,0.2517,17677,4938
effusion,90934,0.4222,0.3932,18694,0.2056,13264,5430


In [5]:
def n_fmt(n):
    """Thousands separator for LaTeX: 85664 -> 85{,}664."""
    return f"${int(n):,}$".replace(",", "{,}")

N_C0_HALF = len(C0_SPLITS["C0_train"]) + len(C0_SPLITS["C0_checkpoint_selection"])

a_rows = [{"Disease": DISEASE_NAMES[d],
           "all_n": n_fmt(r.all_n), "all_pos": f"${100 * r.all_pos:.1f}$",
           "c0_n": n_fmt(r.c0_n), "c0_pos": f"${100 * r.c0_pos:.1f}$", "c0_auc": f"${r.c0_auc:.3f}$",
           "c0_thr": f"${c0_threshold(d):.3f}$",
           "c2_n": n_fmt(r.c2_n), "c2_pos": f"${100 * r.c2_pos:.1f}$", "c2_auc": f"${r.c2_auc:.3f}$"}
          for d, r in c0_A.iterrows()]
m = c0_A.mean()
a_rows.append({"Disease": "Mean", "all_n": "--", "all_pos": "--", "c0_n": "--", "c0_pos": "--", "c0_auc": f"${m.c0_auc:.3f}$", "c0_thr": "--",
               "c2_n": "--", "c2_pos": "--", "c2_auc": f"${m.c2_auc:.3f}$"})

tex = latex_table(
    a_rows, ["Disease", "all_n", "all_pos", "c0_n", "c0_pos", "c0_auc", "c0_thr", "c2_n", "c2_pos", "c2_auc"], group_col="Disease",
    group_rules=False,
    header=["Disease", "Images", "Pos.\\ (\\%)", "Images", "Pos.\\ (\\%)", "AUROC", "Threshold", "Images", "Pos.\\ (\\%)", "AUROC"],
    col_fmt="lrr|rrcc|rrc",
    pre_header=["& \\multicolumn{2}{c|}{All (before split)} & \\multicolumn{4}{c|}{$C_0$ data (train + selection)} "
                "& \\multicolumn{3}{c}{Held-out set ($C_2$ train + test)} \\\\",
                "\\cmidrule(lr){2-3} \\cmidrule(lr){4-7} \\cmidrule(l){8-10}"],
    caption="The data split, per disease: all labelled images, split into the images used to build the disease classifier $C_0$ "
            "(DenseNet-121, multi-label, U-Ignore), and the held-out images, never seen by $C_0$, on which "
            "$C_2$ is trained and tested. AUROC is that of $C_0$, and the threshold is its operating threshold.",
    label="tab:c0_split",
    notes=f"Images: confidently labelled images for that disease; uncertain labels are masked, so counts differ "
          f"by disease. Positive rate and AUROC on those images only. $C_0$ AUROC on the $C_0$ side is measured on "
          f"the selection set ({n_fmt(len(C0_SPLITS['C0_checkpoint_selection']))} images), which is used for early "
          f"stopping and to choose the operating threshold (maximising Youden's $J$) but not for gradient updates. An image is predicted positive when its probability exceeds the threshold. $C_0$ side: "
          f"{n_fmt(N_C0_HALF)} images ({n_fmt(len(C0_SPLITS['C0_train']))} training + "
          f"{n_fmt(len(C0_SPLITS['C0_checkpoint_selection']))} selection); held-out set: "
          f"{n_fmt(len(C0_SPLITS['C2_dataset']))} images. The two are patient-disjoint; frontal views only. "
          "All: the two sides added up.",
).replace("\nMean &", "\n\\midrule\nMean &")
save_table(tex, "c0_split")

wrote /zhome/d0/a/221493/thesis/final_experiment/tables/c0_split.tex

\begin{table}[htbp]
\centering
\small
\caption{The data split, per disease: all labelled images, split into the images used to build the disease classifier $C_0$ (DenseNet-121, multi-label, U-Ignore), and the held-out images, never seen by $C_0$, on which $C_2$ is trained and tested. AUROC is that of $C_0$.}
\label{tab:c0_split}
\begin{threeparttable}
\begin{tabular}{lrr|rrc|rrc}
\toprule
& \multicolumn{2}{c|}{All (before split)} & \multicolumn{3}{c|}{$C_0$ data (train + selection)} & \multicolumn{3}{c}{Held-out set ($C_2$ train + test)} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-6} \cmidrule(l){7-9}
Disease & Images & Pos.\ (\%) & Images & Pos.\ (\%) & AUROC & Images & Pos.\ (\%) & AUROC \\
\midrule
Atelectasis & $161{,}156$ & $18.4$ & $80{,}218$ & $18.2$ & $0.712$ & $80{,}938$ & $18.6$ & $0.715$ \\
Cardiomegaly & $184{,}312$ & $12.7$ & $91{,}805$ & $12.9$ & $0.869$ & $92{,}507$ & $12.5$ & $0.867$ \\
Consolidation & $166

In [6]:
b_rows = [{"Disease": DISEASE_NAMES[d], "n": n_fmt(r.n), "pos": f"${100 * r.pos:.1f}$", "thr": f"${r.threshold:.3f}$",
           "err": f"{n_fmt(r.errors)} (${100 * r.err_rate:.1f}$)", "fp": n_fmt(r.fp), "fn": n_fmt(r.fn)}
          for d, r in c0_B.iterrows()]

tex = latex_table(
    b_rows, ["Disease", "n", "pos", "thr", "err", "fp", "fn"], group_col="Disease", group_rules=False,
    header=["Disease", "Images", "Pos.\\ (\\%)", "Threshold", "Errors (\\%)", "FP", "FN"],
    col_fmt="lrrc|rrr",
    pre_header=["& \\multicolumn{3}{c|}{Held-out set ($C_2$ train + test)} & \\multicolumn{3}{c}{$C_0$ errors} \\\\",
                "\\cmidrule(lr){2-4} \\cmidrule(lr){5-7}"],
    caption="The correctness task $C_0$ hands to $C_2$, per disease. $C_0$ is applied to the held-out set, which it "
            "never saw during training or selection; these images are $C_2$'s training and test data, and "
            "$C_0$'s errors on them are the class $C_2$ learns to detect.",
    label="tab:c2_data",
    notes=f"Images: confidently labelled images for that disease in the held-out set "
          f"({n_fmt(len(C0_SPLITS['C2_dataset']))} images in total; a few lateral-tagged images are dropped). "
          f"$C_2$ is trained and tested on them with {N_FOLDS}-fold patient-disjoint cross-validation, so every "
          "image is a test image exactly once. Threshold: $C_0$'s operating threshold, maximising Youden's $J$ "
          "on the selection set. Errors: images $C_0$ misclassifies at that threshold (FP $+$ FN), as a count "
          "and as a percentage of Images.",
)
save_table(tex, "c2_data")

wrote /zhome/d0/a/221493/thesis/final_experiment/tables/c2_data.tex

\begin{table}[htbp]
\centering
\small
\caption{The correctness task $C_0$ hands to $C_2$, per disease. $C_0$ is applied to the held-out set, which it never saw during training or selection; these images are $C_2$'s training and test data, and $C_0$'s errors on them are the class $C_2$ learns to detect.}
\label{tab:c2_data}
\begin{threeparttable}
\begin{tabular}{lrrc|rrr}
\toprule
& \multicolumn{3}{c|}{Held-out set ($C_2$ train + test)} & \multicolumn{3}{c}{$C_0$ errors} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7}
Disease & Images & Pos.\ (\%) & Threshold & Errors (\%) & FP & FN \\
\midrule
Atelectasis & $80{,}938$ & $18.6$ & $0.189$ & $32{,}809$ ($40.5$) & $29{,}028$ & $3{,}781$ \\
Cardiomegaly & $92{,}507$ & $12.5$ & $0.160$ & $17{,}548$ ($19.0$) & $14{,}769$ & $2{,}779$ \\
Consolidation & $83{,}688$ & $7.8$ & $0.067$ & $32{,}530$ ($38.9$) & $30{,}845$ & $1{,}685$ \\
Edema & $89{,}852$ & $27.8$ & $0.285$ & $22{,}615$ ($

## 0b. The attribute vector

The C1 attributes every attribute model reads (blocks **I**, **C**, **D** of Section 1 are all built on this vector). One row
per attribute family; **Dim.** is the number of columns, and the last row is the dimension of the vector.

* **Demographic**: age, sex and race, each *predicted* from the image by a pretrained model (not read from the record).
* **Geometric**: size and shape of each segmented anatomical region, from the region's mask.
* **Clinical ratios**: seven ratios of geometric attributes (cardiothoracic ratio, lung asymmetry, lung fractions,
  mediastinal ratio). They are computed from the geometric columns when a fold is built (`add_clinical_ratios`), not by
  the C1 script, but they are C1 attributes all the same.
* **Radiomic**: shape and first-order intensity statistics per region (pyradiomics).

The counts are read from the column names of `C2_dataset_attributes.csv` and checked against the attribute columns the
models are actually given (`c2_folds._relevant_cols` on a disease's `c2_data.csv`). The CheXpert labels and the record's own sex and age are not part of it; they are removed before C2. Full column list: `C1_ATTRIBUTE_VECTOR.md`.

In [ ]:
import c2_folds as F

ATTR_CSV = os.path.join("results", "C1_attributes", "C2_dataset_attributes.csv")
head = pd.read_csv(ATTR_CSV, nrows=5)
cols = list(head.columns)

N_REGIONS = len({c[: -len("_area_pixels")] for c in cols if c.endswith("_area_pixels")})
GEO_SUFFIX = ("_area_pixels", "_bbox_width", "_bbox_height", "_bbox_ratio", "_perimeter")
demo  = [c for c in cols if c == "age_pred" or c.startswith(("sex_", "race_"))]
geo   = [c for c in cols if c.endswith(GEO_SUFFIX)]
shape = [c for c in cols if "_original_shape2D_" in c]
first = [c for c in cols if "_original_firstorder_" in c]
ratios = [c for c in F.add_clinical_ratios(head).columns if c not in cols]
groups = {"demo": demo, "geo": geo, "ratio": ratios, "shape": shape, "first": first}
dim = sum(len(v) for v in groups.values())

# What the models actually get: the attribute columns of a disease's c2_data.csv after the ratios are added.
_d = DISEASES[0]
_c2 = pd.read_csv(F.c2_data_path(_d), nrows=5, usecols=lambda c: not c.startswith("emb_"))
_c2 = _c2.rename(columns={"prob": f"{_d}_prob", "pred": f"{_d}_pred", "true": f"{_d}_true"})
model_dim = len(F._relevant_cols(F.add_clinical_ratios(_c2), _d))
assert dim == model_dim, f"table says {dim} attributes, the models are given {model_dim}"
for k in ("geo", "shape", "first"):
    assert len(groups[k]) % N_REGIONS == 0

per = {k: len(groups[k]) // N_REGIONS for k in ("geo", "shape", "first")}
attr_rows = [
    {"fam": "Demographic",      "what": "age; sex (2); race (3)",                      "per": "--",
     "dim": f"${len(demo)}$"},
    {"fam": "Geometric",        "what": "area, bounding-box width / height / ratio, perimeter", "per": f"${per['geo']}$",
     "dim": f"${len(geo)}$"},
    {"fam": "Clinical ratios",  "what": "cardiothoracic, lung asymmetry (3), lung fractions (2), mediastinal",
     "per": "--", "dim": f"${len(ratios)}$"},
    {"fam": "Radiomic: shape",  "what": "elongation, axis lengths, diameter, sphericity, \\dots", "per": f"${per['shape']}$",
     "dim": f"${len(shape)}$"},
    {"fam": "Radiomic: intensity", "what": "mean, median, entropy, skewness, energy, \\dots", "per": f"${per['first']}$",
     "dim": f"${len(first)}$"},
    {"fam": "\\textbf{Total}",  "what": "", "per": "", "dim": f"$\\mathbf{{{dim}}}$"},
]

tex = latex_table(
    attr_rows, ["fam", "what", "per", "dim"], group_col="fam", group_rules=False,
    header=["Attribute family", "Attributes", f"Per region", "Dim."],
    col_fmt="llcr",
    caption=f"Composition of the attribute vector. Each image is described by {dim} attributes: its demographics "
            f"as predicted from the image, the geometry and radiomics of {N_REGIONS} segmented anatomical regions, "
            f"and {len(ratios)} clinical ratios.",
    label="tab:attribute_vector",
    notes=f"Per region: number of attributes extracted for each of the {N_REGIONS} regions (left and right clavicle, "
          "left and right scapula, left and right lung, left and right hilum, heart, diaphragm, mediastinum, "
          "trachea and oesophagus, spine, aorta). A region the segmenter does not find has all its attributes set to $0$. "
          "Geometric attributes are measured on the segmentation mask; radiomic ones on the image under the mask "
          "(pyradiomics, shape in 2D and first-order statistics; no texture). Clinical ratios are computed "
          "from the geometric attributes of the lungs, heart and mediastinum.",
)
# midrule above the total
tex = tex.replace("\\textbf{Total}", "\\midrule\n\\textbf{Total}", 1)
display(pd.DataFrame(attr_rows)[["fam", "what", "per", "dim"]])
save_table(tex, "attribute_vector")


## 1. Attribute models, KNN counterfactuals, K = 1 / 5 / 10

Block ablation from `c2_train_attributes.py` (probability-free MLP): **I** = query attributes, **C** = counterfactual
attributes, **D** = query − counterfactual delta. Results are in
`C2_final_results/<disease>/knn_correct_cf/cf_{K}/attributes/`.

Only K=10 was trained. K=1 and K=5 average the K=10 model's per-counterfactual probabilities over the nearest 1 or 5
counterfactuals (see `run_config.json`), with no refit. **I** uses no counterfactuals, so it has the same value at every K.
CF-anchor is the untrained counterfactual-probability baseline written by the same script.

Cells are the mean ± std of the AUROC for predicting C0 correctness over the 5 CV folds. Bold marks the best entry in each row. MSP and I are printed once per disease, and they are bold if they are best at any $K$.

In [ ]:
ATTR_SOURCE  = "knn_correct_cf"
ATTR_K       = [1, 5, 10]
ATTR_CONFIGS = ["I", "C", "D", "I+C", "I+C+D"]
ATTR_SHOW_ANCHOR = False
ATTR_LABELS  = {"I": "Input (I)", "C": "Counterfactual (Cf)", "D": "Delta (D)",
                "I+C": "I+Cf", "I+C+D": "I+Cf+D", "I+C+D+P": "I+Cf+D+P", "CF-anchor": "CF-anchor"}
# Columns that do not depend on K: printed once per disease instead of once per K.
ATTR_K_INDEPENDENT = ["MSP", "I"]

# Significance of K=10 vs K=1, from section 9 of bootstrapping.ipynb (same draws as the main export;
# Holm across the five diseases within each model). Stars sit on the K=10 cell of each config.
K_DELTAS  = pd.read_csv(os.path.join(C2, "bootstrap_k_deltas.csv"))
_KMETA    = json.load(open(os.path.join(C2, "bootstrap_meta.json")))
K_LEVELS  = [(t, s) for t, s in [(0.001, "***"), (0.01, "**"), (_KMETA["alpha"], "*")]
             if t > 1 / _KMETA["n_boot"]]
K_STARS_KEY = ", ".join(f"{s}~$p_{{adj}} < {t:g}$" for t, s in K_LEVELS)

def k_sig(family, model, disease, hi, lo):
    """Stars of the paired bootstrap `K=hi - K=lo` for one model and disease; "" if not significant."""
    r = K_DELTAS[(K_DELTAS.family == family) & (K_DELTAS.model == model) & (K_DELTAS.disease == disease)
                 & K_DELTAS.comparison.str.endswith(f"K{hi} - K{lo}")]
    assert len(r) == 1, (family, model, disease, len(r))
    p = r.iloc[0].p_adj
    return next((s for t, s in K_LEVELS if p < t), "")

def attr_dir(disease, k, source=ATTR_SOURCE):
    return os.path.join(C2, disease, source, f"cf_{k}", "attributes")

def attribute_table(source=ATTR_SOURCE, ks=ATTR_K, configs=ATTR_CONFIGS, anchor=ATTR_SHOW_ANCHOR):
    """Returns (numeric long frame, formatted rows for LaTeX)."""
    long, rows = [], []
    for d in DISEASES:
        # MSP is K-independent and all K share the same test rows -> compute once.
        msp = msp_fold_aucs(attr_dir(d, max(ks), source), d)
        per_k = {}
        for k in ks:
            folds = summary_fold_aucs(attr_dir(d, k, source))
            entries = {**({"MSP": msp} if SHOW_MSP else {}), **{c: folds[c] for c in configs}}
            if anchor:
                entries["CF-anchor"] = folds["cf_anchor"]
            per_k[k] = entries
        # Best per row; a merged K-independent cell is bold if it is best in any of its rows.
        best = {k: max(e, key=lambda c: e[c].mean()) for k, e in per_k.items()}
        for k, entries in per_k.items():
            row = {"Disease": DISEASE_NAMES[d], "$K$": k}
            for c, f in entries.items():
                bold = (c in best.values()) if c in ATTR_K_INDEPENDENT else (c == best[k])
                indep = c in ATTR_K_INDEPENDENT or c == "CF-anchor"
                sig = "" if indep or k != max(ks) else k_sig("attr", c, d, max(ks), min(ks))
                row[c] = fmt_cell(f, bold=bold, sig=sig)
                long.append({"disease": d, "K": k, "model": c,
                             "auc_mean": f.mean(), "auc_std": f.std()})
            rows.append(row)
    return pd.DataFrame(long), rows

attr_long, attr_rows = attribute_table()
attr_wide = attr_long.pivot_table(index=["disease", "K"], columns="model", values="auc_mean")
attr_wide[[*msp_cols(), *ATTR_CONFIGS] + (["CF-anchor"] if ATTR_SHOW_ANCHOR else [])].round(4)

In [ ]:
models = [*ATTR_CONFIGS] + (["CF-anchor"] if ATTR_SHOW_ANCHOR else [])
cols = ["Disease", "$K$", *msp_cols(), *models]
tex = latex_table(
    attr_rows, cols, group_col="Disease",
    header=["Disease", "$K$", *([MSP_LABEL] if SHOW_MSP else []), *[ATTR_LABELS[m] for m in models]],
    col_fmt=msp_col_fmt(len(models)),  # rule after MSP
    caption="AUROC for predicting C0 correctness with the attribute-based models, using KNN "
            "counterfactuals and $K \\in \\{1,5,10\\}$. Values are mean $\\pm$ std over 5 CV folds. "
            "I: input attributes/embeddings, Cf: counterfactual attributes/embeddings, "
            "D: query$-$counterfactual difference. Stars on the $K=10$ cells: paired patient-level bootstrap of "
            f"$K=10$ against $K=1$ ({_KMETA['n_boot']} replicates), Holm-adjusted across the five diseases "
            f"within each configuration; {K_STARS_KEY}; no star: not significant (I does not depend on $K$).",
    label="tab:attr_knn_k",
    merge_cols=ATTR_K_INDEPENDENT,
)
save_table(tex, "attr_knn_k")

## 2. Image models (CNNs), KNN counterfactuals, K = 1 / 5 / 10

Image models from `c2_train_images.py`. Results are in
`C2_final_results/<disease>/knn_correct_cf/cf_{K}/images/<model>/`. Short names used in the table:

| key | table | input |
|---|---|---|
| `xi` | $x_i$ | query image |
| `xi_sal` | $x_i$+S | query + saliency |
| `cf` | CF | counterfactual image |
| `cf_sal` | CF+S | counterfactual + saliency |
| `dual` | Dual | query + counterfactual (dual encoder) |
| `dual_sal` | Dual+S | dual + saliency |
| `dual_sal_scalars` | Dual+S+Sc | dual + saliency + scalars at the head |

Only K=1 was trained. K=10 evaluates the K=1 models on the 10 nearest counterfactuals and averages the probabilities
(`eval_from` in `run_config.json`). K=5 averages the first 5 of those, with no refit. $x_i$ and $x_i$+S see no
counterfactual, so they are K-independent and printed once per disease, like MSP.

The CNN folds hold the same test rows as the attribute folds (this is asserted below), so the MSP column is the same as in Section 1.
Cells are mean ± std AUROC over the 5 folds. Bold marks the best entry in each row.

In [ ]:
CNN_SOURCE = "knn_correct_cf"
CNN_K      = [1, 5, 10]
CNN_MODELS = ["xi", "xi_sal", "cf", "cf_sal", "dual", "dual_sal"]
CNN_LABELS = {"xi": "Input (I)", "xi_sal": "I + Saliency (S)", "cf": "Cf", "cf_sal": "Cf+S",
              "dual": "Dual (I+Cf)", "dual_sal": "Dual+S", "dual_sal_scalars": "Dual+S+Sc"}
# Models with no counterfactual input: printed once per disease instead of once per K.
CNN_K_INDEPENDENT = ["MSP", "xi", "xi_sal"]

def cnn_dir(disease, k, source=CNN_SOURCE):
    return os.path.join(C2, disease, source, f"cf_{k}", "images")

def cnn_summary_fold_aucs(run_dir):
    """images/cv_summary.csv -> {model key: array of per-fold AUROCs}."""
    s = pd.read_csv(os.path.join(run_dir, "cv_summary.csv"))
    return {r.model: np.array([float(x) for x in r.auc_folds.split(";")]) for r in s.itertuples()}

def check_same_rows(disease, source=CNN_SOURCE):
    """MSP is read from the attribute folds, so the CNN folds must hold exactly the same rows."""
    for i, (a, c) in enumerate(zip(fold_predictions(attr_dir(disease, max(ATTR_K), source)),
                                   fold_predictions(os.path.join(cnn_dir(disease, 1, source), "dual"))),):
        assert set(a["path"]) == set(c["path"]), f"{disease} fold {i}: CNN and attribute test rows differ"

def cnn_table(source=CNN_SOURCE, ks=CNN_K, models=CNN_MODELS):
    """Returns (numeric long frame, formatted rows for LaTeX)."""
    long, rows = [], []
    for d in DISEASES:
        check_same_rows(d, source)
        msp = msp_fold_aucs(attr_dir(d, max(ATTR_K), source), d)
        per_k = {}
        for k in ks:
            folds = cnn_summary_fold_aucs(cnn_dir(d, k, source))
            per_k[k] = {**({"MSP": msp} if SHOW_MSP else {}), **{m: folds[m] for m in models}}
        # Best per row; a merged K-independent cell is bold if it is best in any of its rows.
        best = {k: max(e, key=lambda c: e[c].mean()) for k, e in per_k.items()}
        for k, entries in per_k.items():
            row = {"Disease": DISEASE_NAMES[d], "$K$": k}
            for c, f in entries.items():
                bold = (c in best.values()) if c in CNN_K_INDEPENDENT else (c == best[k])
                sig = "" if c in CNN_K_INDEPENDENT or k != max(ks) else k_sig("cnn", c, d, max(ks), min(ks))
                row[c] = fmt_cell(f, bold=bold, sig=sig)
                long.append({"disease": d, "K": k, "model": c,
                             "auc_mean": f.mean(), "auc_std": f.std()})
            rows.append(row)
    return pd.DataFrame(long), rows

cnn_long, cnn_rows = cnn_table()
cnn_wide = cnn_long.pivot_table(index=["disease", "K"], columns="model", values="auc_mean")
cnn_wide[[*msp_cols(), *CNN_MODELS]].round(4)

In [ ]:
cols = ["Disease", "$K$", *msp_cols(), *CNN_MODELS]
tex = latex_table(
    cnn_rows, cols, group_col="Disease",
    header=["Disease", "$K$", *([MSP_LABEL] if SHOW_MSP else []), *[CNN_LABELS[m] for m in CNN_MODELS]],
    col_fmt=msp_col_fmt(len(CNN_MODELS)),  # rule after MSP
    caption="AUROC for predicting C0 correctness with the image-based models (CNNs), using KNN "
            "counterfactuals and $K \\in \\{1,5,10\\}$. Values are mean $\\pm$ std over 5 CV folds. "
            "I: query image, Cf: counterfactual image, Dual: query and counterfactual (dual encoder), "
            "+S: with saliency map. Stars on the $K=10$ cells: paired patient-level bootstrap of "
            f"$K=10$ against $K=1$ ({_KMETA['n_boot']} replicates), Holm-adjusted across the five diseases "
            f"within each model; {K_STARS_KEY}; no star: not significant (I and I+S do not depend on $K$).",
    label="tab:cnn_knn_k",
    merge_cols=CNN_K_INDEPENDENT,
    resize=True,
)
save_table(tex, "cnn_knn_k")

## 3. Ensemble (attribute model + CNN), KNN counterfactuals, K = 10

One row per disease: the final system of `C2_evaluation.ipynb` (`FINAL`). The members are the ones `c2_ensemble.py`
combined (read from its `run_config.json`): the attribute MLP **I+C+D** and the CNN **Dual+S** (`dual_sal`), both at
K=10 and probability-free (adding C0's probability did not help; Section 1f of `C2_evaluation.ipynb`). Their columns
are their own mean ± std AUROC over the 5 folds, as in Sections 1 and 2.

**The ensemble columns are gains only**, one per combiner: the combiner's mean-fold AUROC minus that of the better
member, i.e. `delta_vs_best` in `ensemble_summary.csv`. Bold marks the largest gain in each row.

* **Mean**: arithmetic mean of the two probabilities. No fitting.
* **Rank**: mean of the two within-fold percentile ranks. No fitting, and not affected by calibration.
* **LR**: logistic regression on the two probabilities, fitted leave-one-fold-out (stacking, so mildly optimistic).

**Significance** comes from `bootstrapping.ipynb`, not from here: the paired, patient-level bootstrap of
`combiner − better member` (comparison `ENS <m> - better component`, the same test Figure 2b annotates), Holm-adjusted
across the 15 combiner × disease tests of that figure. `***` / `**` / `*` is `p_adj` below 0.001 / 0.01 / 0.05; a level
at or below the bootstrap's resolution `1/N_BOOT` is dropped, and no star means not significant. The cell stops if the
bootstrap tested a different K, different members, or a different better member than this table shows.


In [11]:
SHOW_MSP  = True


In [12]:
ENS_SOURCE = "knn_correct_cf"
ENS_METHODS = ["mean", "rank", "lr"]
ENS_LABELS = {"mean": "Mean", "rank": "Rank", "lr": "LR"}

def ens_dir(disease, k, source=ENS_SOURCE):
    return os.path.join(C2, disease, source, f"cf_{k}", "ensemble_prob" if BOOT_META["final"]["prob"] else "ensemble")

def ensemble_fold_aucs(run_dir):
    """Per-fold AUROC of both members and every combiner, from the out-of-fold predictions.
    Returns ({column: fold array}, run_config, oof)."""
    cfg = json.load(open(os.path.join(run_dir, "run_config.json")))
    oof = pd.read_csv(os.path.join(run_dir, "oof_predictions.csv"))
    members = [f"{cfg['attr_model']}_prob", f"{cfg['cnn_model']}_prob"]
    out = {}
    for c in members + [f"ens_{m}" for m in cfg["methods"]]:
        out[c] = np.array([roc_auc_score(g["correct"], g[c]) for _, g in oof.groupby("fold")])
    # Recomputed folds must reproduce the script's own summary.
    s = pd.read_csv(os.path.join(run_dir, "ensemble_summary.csv"))
    for r in s.itertuples():
        col = r.members if r.kind == "component" else f"ens_{r.method}"
        assert np.isclose(out[col].mean(), r.auc_mean, atol=1e-6), (run_dir, col)
    return out, cfg, oof

# Significance, from bootstrapping.ipynb (the Figure 2b comparisons).
BOOT_DELTAS = pd.read_csv(os.path.join(C2, "bootstrap_deltas.csv"))
BOOT_META   = json.load(open(os.path.join(C2, "bootstrap_meta.json")))
assert BOOT_META["msp_variant"] == MSP_VARIANT, (BOOT_META["msp_variant"], MSP_VARIANT)
STAR_LEVELS = [(t, s) for t, s in [(0.001, "***"), (0.01, "**"), (BOOT_META["alpha"], "*")]
               if t > 1 / BOOT_META["n_boot"]]
STARS_KEY   = ", ".join(f"{s}~$p_{{adj}} < {t:g}$" for t, s in STAR_LEVELS)

def stars(p):
    """Stars for a Holm-adjusted p; empty when not significant."""
    return next((s for t, s in STAR_LEVELS if p < t), "")

def ensemble_table(source=ENS_SOURCE, methods=ENS_METHODS):
    """One row per disease at the bootstrap's ensemble K. Returns (numeric frame, LaTeX rows, members)."""
    long, rows, member_names = [], [], set()
    for d in DISEASES:
        pk = BOOT_META["picks"][d]
        k  = pk["ens_k"]
        folds, cfg, oof = ensemble_fold_aucs(ens_dir(d, k, source))
        assert cfg["attr_cf_count"] == cfg["cnn_cf_count"] == k, (d, k, cfg)
        assert set(oof["path"]) == set(pd.concat(fold_predictions(attr_dir(d, max(ATTR_K), source)))["path"]), \
            f"{d}: ensemble rows differ from attribute rows"
        attr, cnn = folds[f"{cfg['attr_model']}_prob"], folds[f"{cfg['cnn_model']}_prob"]
        # The table and the bootstrap must be about the same models.
        attr_name = f"{cfg['attr_model'].split('_', 1)[1]}@K{k}"     # "MLP_I+C+D" -> "I+C+D@K10"
        cnn_name  = f"CNN {cfg['cnn_model']}@K{k}"
        better    = attr_name if attr.mean() >= cnn.mean() else cnn_name
        assert (pk["ens_attr"], pk["ens_cnn"], pk["ens_best_comp"]) == (attr_name, cnn_name, better), \
            f"{d}: bootstrap tested {pk['ens_attr']}, {pk['ens_cnn']} vs {pk['ens_best_comp']} -- re-run bootstrapping.ipynb"
        member_names.add((cfg["attr_model"], cfg["cnn_model"], k))

        scores = {**({"MSP": msp_fold_aucs(attr_dir(d, max(ATTR_K), source), d)} if SHOW_MSP else {}),
                  "attr": attr, "cnn": cnn}
        best   = max(scores, key=lambda c: scores[c].mean())
        row = {"Disease": DISEASE_NAMES[d],
               **{c: fmt_cell(f, bold=(c == best)) for c, f in scores.items()}}
        num = {"disease": d, "K": k, **{c: f.mean() for c, f in scores.items()}}

        gains = {m: folds[f"ens_{m}"].mean() - max(attr.mean(), cnn.mean()) for m in methods}
        top   = max(gains, key=gains.get)
        for m in methods:
            st = BOOT_DELTAS[(BOOT_DELTAS.comparison == f"ENS {m} - better component")
                             & (BOOT_DELTAS.disease == d)]
            assert len(st) == 1, (d, m, len(st))
            st, sig = st.iloc[0], stars(st.iloc[0].p_adj)
            v = f"\\mathbf{{{gains[m]:+.4f}}}" if m == top else f"{gains[m]:+.4f}"
            row[m] = f"${v}^{{{sig}}}$" if sig else f"${v}$"
            num.update({f"{m} gain": gains[m], f"{m} boot lo": st.lo, f"{m} boot hi": st.hi,
                        f"{m} p_adj": st.p_adj})
        rows.append(row)
        long.append(num)
    assert len(member_names) == 1, f"members differ across diseases: {member_names}"
    return pd.DataFrame(long).set_index("disease"), rows, member_names.pop()

ens_tab, ens_rows, ENS_MEMBERS = ensemble_table()
print("members:", ENS_MEMBERS,
      f"| bootstrap: {BOOT_META['n_boot']} replicates, written {BOOT_META['created']}")
ens_tab.round(4)


members: ('MLP_I+C+D', 'dual_sal', 10) | bootstrap: 5000 replicates, written 2026-10-02T12:05:48


,K,MSP,attr,cnn,mean gain,mean boot lo,mean boot hi,mean p_adj,rank gain,rank boot lo,rank boot hi,rank p_adj,lr gain,lr boot lo,lr boot hi,lr p_adj
disease,,,,,,,,,,,,,,,,
atelectasis,10,0.8633,0.8617,0.8562,0.0002,-0.0005,0.0010,0.5416,0.0005,-0.0002,0.0012,0.3544,0.0009,0.0004,0.0014,0.0030
cardiomegaly,10,0.9168,0.9188,0.9163,0.0016,0.0011,0.0022,0.0030,0.0017,0.0011,0.0022,0.0030,0.0018,0.0014,0.0022,0.0030
consolidation,10,0.9462,0.9447,0.9424,0.0006,0.0000,0.0011,0.1176,0.0008,0.0003,0.0013,0.0040,0.0006,0.0001,0.0011,0.0608
edema,10,0.8303,0.8344,0.8314,0.0027,0.0021,0.0034,0.0030,0.0027,0.0020,0.0033,0.0030,0.0028,0.0023,0.0034,0.0030
effusion,10,0.7817,0.7826,0.7776,0.0046,0.0035,0.0057,0.0030,0.0047,0.0036,0.0057,0.0030,0.0049,0.0040,0.0057,0.0030


In [13]:
ENS_ATTR, ENS_CNN, ENS_K = ENS_MEMBERS
cols = ["Disease", *msp_cols(), "attr", "cnn", *ENS_METHODS]
tex = latex_table(
    ens_rows, cols, group_col="Disease", group_rules=False,
    header=["Disease", *([MSP_LABEL] if SHOW_MSP else []),
            ATTR_LABELS[ENS_ATTR.split("_", 1)[1]], CNN_LABELS[ENS_CNN],
            *[ENS_LABELS[m] for m in ENS_METHODS]],
    col_fmt="l" + ("c|" if SHOW_MSP else "") + "cc|" + "c" * len(ENS_METHODS),
    pre_header=["\\multicolumn{%d}{c}{} & \\multicolumn{%d}{c}{Ensemble gain} \\\\"
                % (3 + SHOW_MSP, len(ENS_METHODS)),
                "\\cmidrule(l){%d-%d}" % (4 + SHOW_MSP, 3 + SHOW_MSP + len(ENS_METHODS))],
    caption=f"AUROC for predicting C0 correctness at $K={ENS_K}$ (KNN counterfactuals) of the attribute model "
            f"({ATTR_LABELS[ENS_ATTR.split('_', 1)[1]]}) and the CNN ({CNN_LABELS[ENS_CNN]}), and the gain of "
            f"ensembling them with each of three combiners. Member values are mean $\\pm$ std over 5 CV folds.",
    label="tab:ens_knn",
    notes=f"Ensemble gain: mean-fold AUROC of the ensemble minus that of the better member; bold is the "
          f"largest gain per row. Mean: average of the probabilities; Rank: average of within-fold percentile "
          f"ranks; LR: logistic-regression stacking fitted leave-one-fold-out. Stars: paired patient-level bootstrap of that gain ({BOOT_META['n_boot']} "
          f"replicates), Holm-adjusted across the five diseases within each combiner; {STARS_KEY}; no star: not significant.",
)
save_table(tex, "ens_knn")


wrote /zhome/d0/a/221493/thesis/final_experiment/tables/ens_knn.tex

\begin{table}[htbp]
\centering
\small
\caption{AUROC for predicting C0 correctness at $K=10$ (KNN counterfactuals) of the attribute model (I+Cf+D) and the CNN (Dual+S), and the gain of ensembling them with each of three combiners. Member values are mean $\pm$ std over 5 CV folds.}
\label{tab:ens_knn}
\begin{threeparttable}
\begin{tabular}{lc|cc|ccc}
\toprule
\multicolumn{4}{c}{} & \multicolumn{3}{c}{Ensemble gain} \\
\cmidrule(l){5-7}
Disease & Adjusted MSP & I+Cf+D & Dual+S & Mean & Rank & LR \\
\midrule
Atelectasis & $\mathbf{0.863} \pm 0.004$ & $0.862 \pm 0.005$ & $0.856 \pm 0.006$ & $+0.0002$ & $+0.0005$ & $\mathbf{+0.0009}^{**}$ \\
Cardiomegaly & $0.917 \pm 0.002$ & $\mathbf{0.919} \pm 0.003$ & $0.916 \pm 0.003$ & $+0.0016^{**}$ & $+0.0017^{**}$ & $\mathbf{+0.0018}^{**}$ \\
Consolidation & $\mathbf{0.946} \pm 0.003$ & $0.945 \pm 0.003$ & $0.942 \pm 0.002$ & $+0.0006$ & $\mathbf{+0.0008}^{**}$ & $+0.0006$ \\
Edema

## 3b. Attribute model vs CNN, KNN counterfactuals, K = 10

Claim 7 of `bootstrapping.ipynb` (`best attr - best CNN`): which modality reads C0's correctness better? The two models
are the final system's (`FINAL`): the attribute MLP **I+C+D** and the CNN **Dual+S** (`dual_sal`), both at K=10 and
probability-free. They are the members of the Section 3 ensemble.

Model cells are mean ± std AUROC over the 5 folds. Bold marks the better model in each row. **Δ** is attribute − CNN
(mean-fold AUROC). Stars come from `bootstrapping.ipynb`
(group `claims`): a paired, patient-level bootstrap of Δ, Holm-adjusted across the five diseases within this
comparison. The Mean row averages the five diseases' mean-fold AUROCs (no std). Its stars come from the bootstrap's
`AVERAGE` row, a single test.

In [ ]:
AC_K         = BOOT_META["final"]["k"]
AC_ATTR      = BOOT_META["final"]["attr"]       # "I+C+D"
AC_CNN       = BOOT_META["final"]["cnn"]        # "dual_sal"
AC_SHOW_MEAN = True    # Mean-over-diseases row in the LaTeX table (always in the frame)
AC_COMPARISON = "best attr - best CNN"
assert not BOOT_META["final"]["prob"], "the final system uses C0's probability -- point the runs below at *_prob"

def ac_fold_aucs(d, k=AC_K, source=ENS_SOURCE):
    """Per-fold AUROC of the attribute model and the CNN, from the fold predictions (not the
    rounded cv_summary.csv), on the same test rows in the same folds."""
    attr_f = fold_predictions(attr_dir(d, k, source))
    cnn_f  = fold_predictions(os.path.join(cnn_dir(d, k, source), AC_CNN))
    assert [set(f["path"]) for f in attr_f] == [set(f["path"]) for f in cnn_f], f"{d}: test rows differ"
    return (np.array([roc_auc_score(f["correct"], f[f"MLP_{AC_ATTR}_prob"]) for f in attr_f]),
            np.array([roc_auc_score(f["correct"], f[f"{AC_CNN}_prob"]) for f in cnn_f]))

def ac_boot(d):
    r = BOOT_DELTAS[(BOOT_DELTAS.group == "claims") & (BOOT_DELTAS.comparison == AC_COMPARISON)
                    & (BOOT_DELTAS.disease == d)]
    assert len(r) == 1, (d, len(r))
    return r.iloc[0]

def delta_cell(delta, p_adj):
    sig = stars(p_adj)
    return f"${delta:+.4f}^{{{sig}}}$" if sig else f"${delta:+.4f}$"

def attr_cnn_table():
    long, rows = [], []
    for d in DISEASES:
        # The bootstrap must have compared these two models.
        pk = BOOT_META["picks"][d]
        assert (pk["best_attr"], pk["best_cnn"]) == (f"{AC_ATTR}@K{AC_K}", f"CNN {AC_CNN}@K{AC_K}"), \
            f"{d}: bootstrap compared {pk['best_attr']} vs {pk['best_cnn']} -- re-run bootstrapping.ipynb"
        attr, cnn = ac_fold_aucs(d)
        delta = attr.mean() - cnn.mean()
        b = ac_boot(d)
        # Delta is the observed difference; the bootstrap's `delta` is its mean over replicates.
        assert abs(b.delta - delta) < 1e-4, (d, b.delta, delta)
        rows.append({"Disease": DISEASE_NAMES[d],
                     "attr": fmt_cell(attr, bold=attr.mean() >= cnn.mean()),
                     "cnn":  fmt_cell(cnn,  bold=cnn.mean() > attr.mean()),
                     "d": delta_cell(delta, b.p_adj)})
        long.append({"disease": d, "attr": attr.mean(), "cnn": cnn.mean(), "delta": delta,
                     "boot lo": b.lo, "boot hi": b.hi, "p_adj": b.p_adj})
    tab = pd.DataFrame(long).set_index("disease")
    # Mean over diseases: no std, stars and interval from the bootstrap's AVERAGE row.
    m, b = tab.mean(), ac_boot("AVERAGE")
    assert abs(b.delta - m["delta"]) < 1e-4, (b.delta, m["delta"])
    tab.loc["MEAN"] = [m["attr"], m["cnn"], m["delta"], b.lo, b.hi, b.p_adj]
    if AC_SHOW_MEAN:
        rows.append({"Disease": "Mean",
                     "attr": f"$\\mathbf{{{m['attr']:.3f}}}$" if m["attr"] >= m["cnn"] else f"${m['attr']:.3f}$",
                     "cnn":  f"$\\mathbf{{{m['cnn']:.3f}}}$" if m["cnn"] > m["attr"] else f"${m['cnn']:.3f}$",
                     "d": delta_cell(m["delta"], b.p_adj)})
    return tab, rows

ac_tab, ac_rows = attr_cnn_table()
ac_tab.round(4)

In [ ]:
tex = latex_table(
    ac_rows, ["Disease", "attr", "cnn", "d"], group_col="Disease", group_rules=False,
    header=["Disease", ATTR_LABELS[AC_ATTR], CNN_LABELS[AC_CNN], "$\\Delta$"],
    col_fmt="lcc|r",
    pre_header=["& \\multicolumn{1}{c}{Attribute model} & \\multicolumn{1}{c|}{CNN} & \\multicolumn{1}{c}{Attribute $-$ CNN} \\\\",
                "\\cmidrule(lr){2-2} \\cmidrule(lr){3-3} \\cmidrule(l){4-4}"],
    caption=f"The attribute model ({ATTR_LABELS[AC_ATTR]}) against the CNN ({CNN_LABELS[AC_CNN]}) at $K={AC_K}$ "
            f"(KNN counterfactuals). AUROC for predicting C0 correctness, mean $\\pm$ std over 5 CV folds.",
    label="tab:attr_vs_cnn_knn",
    notes="Bold: the better model per row. $\\Delta$: mean-fold AUROC of the attribute model minus that of the CNN. "
          f"Stars: paired patient-level bootstrap of $\\Delta$ ({BOOT_META['n_boot']} replicates), Holm-adjusted across the five diseases"
          + ("; for the Mean row, the bootstrap of the five-disease average, a single test" if AC_SHOW_MEAN else "")
          + f"; {STARS_KEY}; no star: not significant."
          + (" Mean: average over the five diseases." if AC_SHOW_MEAN else ""),
)
save_table(tex, "attr_vs_cnn_knn")

## 4. Adding C0's probability to the best models, K = 10

The test of Section 1f of `C2_evaluation.ipynb`: each family's best probability-free model at K=10 (attribute MLP
**I+C+D**, CNN **Dual+S**) is retrained with C0's five probability scalars `[p_xi, H(p_xi), p_cf, H(p_cf), p_xi - p_cf]`
added (**P**). The attribute model gets them as extra inputs (`c2_train_attributes.py --with-prob`, results in
`cf_10/attributes_prob/`); the CNN gets them at the head (`dual_sal_scalars`, in `cf_10/images/`).

Model cells are mean ± std AUROC over the 5 folds. No bold: the pairs mostly agree to 3 decimals, so Δ carries the comparison. **Δ** is with − without
(mean-fold AUROC). Stars come from `bootstrapping.ipynb` (group `prob`): paired patient-level bootstrap of Δ,
Holm-adjusted across the five diseases within a family; the Mean row is a single test. The Mean row
averages the five diseases' mean-fold AUROCs (no std).

In [14]:
SHOW_MSP = False   # no MSP column in this table
PROB_SHOW_MEAN = False   # Mean-over-diseases row in the LaTeX table (always in the frame)

PROB_K = BOOT_META["final"]["k"]
PROB_PAIRS = {"attr": tuple(BOOT_META["prob_pairs"]["attr"]),   # (with, without)
              "cnn":  tuple(BOOT_META["prob_pairs"]["cnn"])}
CNN_LABELS["dual_sal_scalars"] = "Dual+S+P"   # the scalars are C0's probabilities: same P as the attribute side

def prob_fold_aucs(d, k=PROB_K, source=ENS_SOURCE):
    """Per-fold AUROC of the four models: {(family, "with"/"without"): fold array}. Recomputed from
    the fold predictions, not cv_summary.csv, whose 4-decimal rounding shifts Delta off the bootstrap's."""
    (ah, al), (ch, cl) = PROB_PAIRS["attr"], PROB_PAIRS["cnn"]
    base = os.path.join(C2, d, source, f"cf_{k}")
    runs = {("attr", "without"): (os.path.join(base, "attributes"),           f"MLP_{al}_prob"),
            ("attr", "with"):    (os.path.join(base, "attributes_prob"),      f"MLP_{ah}_prob"),
            ("cnn", "without"):  (os.path.join(base, "images", cl),           f"{cl}_prob"),
            ("cnn", "with"):     (os.path.join(base, "images", ch),           f"{ch}_prob")}
    ref = [set(f["path"]) for f in fold_predictions(attr_dir(d, k, source))]
    out = {}
    for key, (run, col) in runs.items():
        folds = fold_predictions(run)
        # Paired comparison: every run must test on exactly the same rows.
        assert [set(f["path"]) for f in folds] == ref, f"{d} {key}: test rows differ"
        out[key] = np.array([roc_auc_score(f["correct"], f[col]) for f in folds])
    return out

def prob_comparison(fam):
    hi, lo = PROB_PAIRS[fam]
    return f"{hi} - {lo}" if fam == "attr" else f"CNN {hi} - {lo}"

def prob_boot(fam, d):
    r = BOOT_DELTAS[(BOOT_DELTAS.group == "prob") & (BOOT_DELTAS.comparison == prob_comparison(fam))
                    & (BOOT_DELTAS.disease == d)]
    assert len(r) == 1, (fam, d, len(r))
    return r.iloc[0]

def delta_cell(delta, p_adj):
    sig = stars(p_adj)
    return f"${delta:+.4f}^{{{sig}}}$" if sig else f"${delta:+.4f}$"

def prob_table():
    long, rows, means = [], [], []
    for d in DISEASES:
        f = prob_fold_aucs(d)
        msp = msp_fold_aucs(attr_dir(d, PROB_K), d)
        row, num = {"Disease": DISEASE_NAMES[d]}, {"disease": d}
        if SHOW_MSP:
            row["MSP"], num["MSP"] = fmt_cell(msp), msp.mean()
        for fam in ["attr", "cnn"]:
            lo, hi = f[(fam, "without")], f[(fam, "with")]
            delta = hi.mean() - lo.mean()
            b = prob_boot(fam, d)
            # Delta is the observed difference; the bootstrap's `delta` is its mean over replicates.
            assert abs(b.delta - delta) < 1e-4, (fam, d, b.delta, delta)
            row[f"{fam}_lo"], row[f"{fam}_hi"] = fmt_cell(lo), fmt_cell(hi)
            row[f"{fam}_d"]  = delta_cell(delta, b.p_adj)
            num.update({f"{fam} without": lo.mean(), f"{fam} with": hi.mean(), f"{fam} delta": delta,
                        f"{fam} boot lo": b.lo, f"{fam} boot hi": b.hi, f"{fam} p_adj": b.p_adj})
        rows.append(row); long.append(num)
    tab = pd.DataFrame(long).set_index("disease")
    # Mean over diseases: no std, stars from the bootstrap's AVERAGE rows.
    m = tab.mean(numeric_only=True)
    tab.loc["MEAN"] = m          # bootstrap columns are overwritten below with the AVERAGE rows
    row = {"Disease": "Mean"}
    if SHOW_MSP:
        row["MSP"] = f"${m['MSP']:.3f}$"
    for fam in ["attr", "cnn"]:
        lo, hi = m[f"{fam} without"], m[f"{fam} with"]
        b = prob_boot(fam, "AVERAGE")
        assert abs(b.delta - (hi - lo)) < 1e-4, (fam, b.delta, hi - lo)
        row[f"{fam}_lo"], row[f"{fam}_hi"] = f"${lo:.3f}$", f"${hi:.3f}$"
        row[f"{fam}_d"]  = delta_cell(hi - lo, b.p_adj)
        tab.loc["MEAN", [f"{fam} boot lo", f"{fam} boot hi", f"{fam} p_adj"]] = [b.lo, b.hi, b.p_adj]
    if PROB_SHOW_MEAN:
        rows.append(row)
    return tab, rows

prob_tab, prob_rows = prob_table()
prob_tab.round(4)

,attr without,attr with,attr delta,attr boot lo,attr boot hi,attr p_adj,cnn without,cnn with,cnn delta,cnn boot lo,cnn boot hi,cnn p_adj
disease,,,,,,,,,,,,
atelectasis,0.8617,0.8613,-0.0004,-0.0009,0.0000,0.2640,0.8562,0.8589,0.0027,0.0017,0.0038,0.0010
cardiomegaly,0.9188,0.9189,0.0000,-0.0004,0.0005,0.9616,0.9163,0.9163,0.0000,-0.0010,0.0010,1.0000
consolidation,0.9447,0.9449,0.0002,-0.0003,0.0006,0.9616,0.9424,0.9424,0.0000,-0.0008,0.0008,1.0000
edema,0.8344,0.8338,-0.0006,-0.0012,0.0001,0.2712,0.8314,0.8316,0.0002,-0.0009,0.0013,1.0000
effusion,0.7826,0.7817,-0.0009,-0.0016,-0.0003,0.0120,0.7776,0.7762,-0.0014,-0.0030,0.0001,0.3088
MEAN,0.8684,0.8681,-0.0004,-0.0006,-0.0001,0.0056,0.8648,0.8651,0.0003,-0.0002,0.0008,0.2364


In [15]:
(ah, al), (ch, cl) = PROB_PAIRS["attr"], PROB_PAIRS["cnn"]
fam_cols = ["attr_lo", "attr_hi", "attr_d", "cnn_lo", "cnn_hi", "cnn_d"]
cols = ["Disease", *msp_cols(), *fam_cols]
off = 1 + SHOW_MSP   # columns before the first family
tex = latex_table(
    prob_rows, cols, group_col="Disease", group_rules=False,
    header=["Disease", *([MSP_LABEL] if SHOW_MSP else []),
            ATTR_LABELS[al], ATTR_LABELS[ah], "$\\Delta$", CNN_LABELS[cl], CNN_LABELS[ch], "$\\Delta$"],
    col_fmt="l" + ("c|" if SHOW_MSP else "") + "ccr|ccr",
    pre_header=["\\multicolumn{%d}{c}{} & \\multicolumn{3}{c|}{Attribute model} & \\multicolumn{3}{c}{CNN} \\\\" % off,
                "\\cmidrule(lr){%d-%d} \\cmidrule(l){%d-%d}" % (off + 1, off + 3, off + 4, off + 6)],
    caption=f"Effect of adding C0's probability (P) to the best attribute model ({ATTR_LABELS[al]}) and the best "
            f"CNN ({CNN_LABELS[cl]}) at $K={PROB_K}$ (KNN counterfactuals). AUROC for predicting C0 correctness, "
            f"mean $\\pm$ std over 5 CV folds.",
    label="tab:prob_knn",
    notes="P: C0's five probability scalars $[p(x_i), H(p(x_i)), p(x_{cf}), H(p(x_{cf})), p(x_i) - p(x_{cf})]$, "
          "given to the attribute model as extra inputs and to the CNN at its classification head. "
          "$\\Delta$: mean-fold AUROC with P minus without. "
          f"Stars: paired patient-level bootstrap of $\\Delta$ ({BOOT_META['n_boot']} replicates), Holm-adjusted "
          f"across the five diseases within each model family"
          + ("; the Mean row is a single test" if PROB_SHOW_MEAN else "")
          + f"; {STARS_KEY}; no star: not significant."
          + (" Mean: average over the five diseases." if PROB_SHOW_MEAN else ""),
)
save_table(tex, "prob_knn")

wrote /zhome/d0/a/221493/thesis/final_experiment/tables/prob_knn.tex

\begin{table}[htbp]
\centering
\small
\caption{Effect of adding C0's probability (P) to the best attribute model (I+Cf+D) and the best CNN (Dual+S) at $K=10$ (KNN counterfactuals). AUROC for predicting C0 correctness, mean $\pm$ std over 5 CV folds.}
\label{tab:prob_knn}
\begin{threeparttable}
\begin{tabular}{lccr|ccr}
\toprule
\multicolumn{1}{c}{} & \multicolumn{3}{c|}{Attribute model} & \multicolumn{3}{c}{CNN} \\
\cmidrule(lr){2-4} \cmidrule(l){5-7}
Disease & I+Cf+D & I+Cf+D+P & $\Delta$ & Dual+S & Dual+S+P & $\Delta$ \\
\midrule
Atelectasis & $0.862 \pm 0.005$ & $0.861 \pm 0.005$ & $-0.0004$ & $0.856 \pm 0.006$ & $0.859 \pm 0.004$ & $+0.0027^{**}$ \\
Cardiomegaly & $0.919 \pm 0.003$ & $0.919 \pm 0.003$ & $+0.0000$ & $0.916 \pm 0.003$ & $0.916 \pm 0.003$ & $+0.0000$ \\
Consolidation & $0.945 \pm 0.003$ & $0.945 \pm 0.003$ & $+0.0002$ & $0.942 \pm 0.002$ & $0.942 \pm 0.002$ & $+0.0000$ \\
Edema & $0.834 \pm 0.004$ &

## 5. Diffusion vs KNN counterfactuals, final models, 20k subset, K = 1

The paired comparison of Section 2c of `C2_evaluation_diffusion.ipynb` (figure `diffusion_aim2c_arm_points`). Both arms
use the same 20,000 images, patient-grouped folds and seed; only the source of the counterfactual differs. Each model sees
one counterfactual (K=1): the high diffusion edit (`diffusion_hml_sub20k`) or the nearest retrieved neighbour
(`knn_correct_cf_sub20k`). The models are the final ones: attribute MLP **I+C+D**, CNN **Dual+S**, and their
ensemble with **each arm's own best combiner** (highest mean-fold AUROC). All three are read from each arm's
`cf_1/ensemble/oof_predictions.csv`, so they are scored on the same rows.

Cells are mean-fold AUROC; std is left out so that nine columns fit the page. **Δ** is diffusion − KNN. Stars come from
`bootstrap_diffusion_external.csv` (section 4 of `bootstrapping_diffusion.ipynb`), the same brackets the figure draws.
Holm runs across the five diseases within each comparison, as everywhere else. The ensemble comparison is between two winners chosen after scoring.

In [16]:
ARM_K       = 1
ARM_SOURCES = {"KNN": "knn_correct_cf_sub20k", "diffusion": "diffusion_hml_sub20k"}
ARM_EXT     = pd.read_csv(os.path.join(C2, "bootstrap_diffusion_external.csv"))
ARM_META    = json.load(open(os.path.join(C2, "bootstrap_diffusion_meta.json")))
# Holm family sizes per disease, for the notes: matched configs (attr + CNN) and best-model comparisons.
ARM_N_HOLM  = ARM_EXT[ARM_EXT.disease == DISEASES[0]].group.value_counts().to_dict()
# (family, header, bootstrap comparison in the external export)
ARM_FAMILIES = [("attr", "Attribute model (I+Cf+D)", "attr I+C+D"),
                ("cnn",  "CNN (Dual+S)",             "CNN dual_sal"),
                ("ens",  "Ensemble",                 "best ens (each arm's own)")]

def arm_fold_aucs(d, arm, k=ARM_K):
    """Per-fold AUROC of the attribute model, the CNN and the arm's best combiner, plus the
    combiner's name and the OOF frame (for the row check)."""
    run = os.path.join(C2, d, ARM_SOURCES[arm], f"cf_{k}", "ensemble")
    cfg = json.load(open(os.path.join(run, "run_config.json")))
    assert (cfg["attr_model"], cfg["cnn_model"], cfg["attr_cf_count"], cfg["cnn_cf_count"]) == \
        ("MLP_I+C+D", "dual_sal", k, k), (d, arm, cfg)
    oof = pd.read_csv(os.path.join(run, "oof_predictions.csv"))
    auc = lambda col: np.array([roc_auc_score(g["correct"], g[col]) for _, g in oof.groupby("fold")])
    ens = {m: auc(f"ens_{m}") for m in cfg["methods"]}
    best = max(ens, key=lambda m: ens[m].mean())
    return {"attr": auc("MLP_I+C+D_prob"), "cnn": auc("dual_sal_prob"), "ens": ens[best]}, best, oof

def arm_table():
    long, rows = [], []
    for d in DISEASES:
        (fk, bk, ok), (fd, bd, od) = arm_fold_aucs(d, "KNN"), arm_fold_aucs(d, "diffusion")
        # Paired comparison: same rows in the same folds.
        assert set(zip(ok["path"], ok["fold"])) == set(zip(od["path"], od["fold"])), f"{d}: arms differ"
        row, num = {"Disease": DISEASE_NAMES[d]}, {"disease": d, "n": len(ok),
                                                   "ens KNN combiner": bk, "ens diffusion combiner": bd}
        for fam, _, cmp in ARM_FAMILIES:
            knn, dif = fk[fam].mean(), fd[fam].mean()
            r = ARM_EXT[(ARM_EXT.comparison == cmp) & (ARM_EXT.disease == d)]
            assert len(r) == 1, (d, cmp, len(r))
            r = r.iloc[0]
            # Delta is the observed difference; the export's `delta` is a bootstrap summary of it.
            assert abs(r.delta - (dif - knn)) < 5e-4, (d, cmp, r.delta, dif - knn)
            row[f"{fam}_knn"], row[f"{fam}_dif"] = f"${knn:.3f}$", f"${dif:.3f}$"
            row[f"{fam}_d"] = delta_cell(dif - knn, r.p_adj)
            num.update({f"{fam} KNN": knn, f"{fam} diffusion": dif, f"{fam} delta": dif - knn,
                        f"{fam} boot lo": r.lo, f"{fam} boot hi": r.hi, f"{fam} p_adj": r.p_adj})
        rows.append(row); long.append(num)
    return pd.DataFrame(long).set_index("disease"), rows

arm_tab, arm_rows = arm_table()
arm_tab.round(4)

,n,ens KNN combiner,ens diffusion combiner,attr KNN,attr diffusion,attr delta,attr boot lo,attr boot hi,attr p_adj,cnn KNN,...,cnn delta,cnn boot lo,cnn boot hi,cnn p_adj,ens KNN,ens diffusion,ens delta,ens boot lo,ens boot hi,ens p_adj
disease,,,,,,,,,,,,,,,,,,,,,
atelectasis,20000,rank,rank,0.8484,0.8546,0.0062,0.0032,0.0092,0.0024,0.8471,...,0.0101,0.0070,0.0133,0.0024,0.8527,0.8602,0.0075,0.0047,0.0102,0.0006
cardiomegaly,20000,rank,lr,0.9034,0.9126,0.0092,0.0067,0.0117,0.0024,0.9019,...,0.0038,0.0011,0.0065,0.0160,0.9078,0.9137,0.0059,0.0037,0.0082,0.0006
consolidation,20000,rank,rank,0.9376,0.9404,0.0028,0.0008,0.0050,0.0608,0.9368,...,0.0041,0.0018,0.0064,0.0072,0.9399,0.9434,0.0035,0.0014,0.0055,0.0024
edema,20000,rank,rank,0.8145,0.8265,0.0121,0.0088,0.0153,0.0024,0.8196,...,0.0070,0.0042,0.0098,0.0024,0.8259,0.8323,0.0064,0.0040,0.0087,0.0006
effusion,20000,rank,rank,0.7642,0.7717,0.0075,0.0033,0.0118,0.0096,0.7633,...,-0.0001,-0.0039,0.0037,1.0000,0.7762,0.7792,0.0029,-0.0002,0.0061,0.1232


In [17]:
COMBINER = {"mean": "Mean", "rank": "Rank", "lr": "LR"}
fam_cols = [f"{fam}_{s}" for fam, _, _ in ARM_FAMILIES for s in ["knn", "dif", "d"]]
combiners = "; ".join(
    f"{DISEASE_NAMES[d]} {COMBINER[r['ens KNN combiner']]}/{COMBINER[r['ens diffusion combiner']]}"
    for d, r in arm_tab.iterrows())
tex = latex_table(
    arm_rows, ["Disease", *fam_cols], group_col="Disease", group_rules=False,
    header=["Disease", *["KNN", "Diff.", "$\\Delta$"] * len(ARM_FAMILIES)],
    col_fmt="l" + "|".join(["ccr"] * len(ARM_FAMILIES)),
    pre_header=[" & " + " & ".join(f"\\multicolumn{{3}}{{c{'|' if i < len(ARM_FAMILIES) - 1 else ''}}}{{{h}}}"
                                   for i, (_, h, _) in enumerate(ARM_FAMILIES)) + " \\\\",
                " ".join(f"\\cmidrule({'lr' if i < len(ARM_FAMILIES) - 1 else 'l'}){{{2 + 3 * i}-{4 + 3 * i}}}"
                         for i in range(len(ARM_FAMILIES)))],
    caption=f"Diffusion versus retrieved (KNN) counterfactuals for the final models, at $K={ARM_K}$ on the same "
            f"20{{,}}000-image subset, folds and seed. AUROC for predicting C0 correctness, mean over 5 CV folds.",
    label="tab:diffusion_vs_knn",
    notes="KNN: the nearest retrieved neighbour with the opposite C0 prediction; Diff.: the strongest (high) "
          "diffusion edit of the query. Ensemble: the attribute model and the CNN combined with each arm's own "
          f"best combiner (KNN/diffusion: {combiners}). $\\Delta$: diffusion minus KNN. "
          f"Stars: paired patient-level bootstrap of $\\Delta$ ({ARM_META['n_boot']} replicates), Holm-adjusted across "
          f"the five diseases within each model; {STARS_KEY}; no star: not significant. The ensemble comparison is between "
          "two combiners each chosen after scoring.",
)
save_table(tex, "diffusion_vs_knn")

wrote /zhome/d0/a/221493/thesis/final_experiment/tables/diffusion_vs_knn.tex

\begin{table}[htbp]
\centering
\small
\caption{Diffusion versus retrieved (KNN) counterfactuals for the final models, at $K=1$ on the same 20{,}000-image subset, folds and seed. AUROC for predicting C0 correctness, mean over 5 CV folds.}
\label{tab:diffusion_vs_knn}
\begin{threeparttable}
\begin{tabular}{lccr|ccr|ccr}
\toprule
 & \multicolumn{3}{c|}{Attribute model (I+Cf+D)} & \multicolumn{3}{c|}{CNN (Dual+S)} & \multicolumn{3}{c}{Ensemble} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(l){8-10}
Disease & KNN & Diff. & $\Delta$ & KNN & Diff. & $\Delta$ & KNN & Diff. & $\Delta$ \\
\midrule
Atelectasis & $0.848$ & $0.855$ & $+0.0062^{**}$ & $0.847$ & $0.857$ & $+0.0101^{**}$ & $0.853$ & $0.860$ & $+0.0075^{***}$ \\
Cardiomegaly & $0.903$ & $0.913$ & $+0.0092^{**}$ & $0.902$ & $0.906$ & $+0.0038^{*}$ & $0.908$ & $0.914$ & $+0.0059^{***}$ \\
Consolidation & $0.938$ & $0.940$ & $+0.0028$ & $0.937$ & $0.941$ & 

## 5b. K = 1 vs K = 3, both arms, 20k subset

Does adding counterfactuals help? On the full set KNN is swept K = 1 / 5 / 10, but diffusion only exists at K = 1 / 3 on
the 20k subset, so the two arms are only comparable there: same 20,000 images, folds and seed, the same final models
(attribute MLP **I+C+D**, CNN **Dual+S**, and their ensemble) as Section 5. K means different things in the two arms:
for KNN it is the number of nearest retrieved neighbours, for diffusion the number of intensity levels pooled
(K=1 is the `high` edit; K=3 adds `mid` and `low`). Both K=1 pairings are the prefix of their K=3 ones.

All three models are read from each run's `cf_{K}/ensemble/oof_predictions.csv`. The ensemble uses **the same combiner
at both K**: the arm's best at K=1 (the one Section 5 reports), so its Δ reflects only the change in K, not a change of
combiner.

Cells are mean-fold AUROC. **Δ** is K=3 − K=1, bold when significant. Stars come from `bootstrap_diffusion_k_deltas.csv` (section 9 of
`bootstrapping_diffusion.ipynb`): a paired, patient-level bootstrap of Δ on the external export's draws, Holm-adjusted
across the five diseases within each arm and model. The Mean row averages the five diseases' mean-fold AUROCs; its
stars are the bootstrap of the five-disease average, a single test. The cell
stops if the bootstrap tested a different delta or combiner than this table shows.

In [ ]:
KARM_K = (1, 3)
KARM_SHOW_MEAN = True   # Mean-over-diseases rows in the LaTeX table (always in the frame)
# Significance, from section 9 of bootstrapping_diffusion.ipynb: paired patient bootstrap of K=3 - K=1.
KARM_BOOT = pd.read_csv(os.path.join(C2, "bootstrap_diffusion_k_deltas.csv")).set_index(["disease", "arm", "model"])

def karm_fold_aucs(d, arm, k, combiner):
    """Per-fold AUROC of attr I+C+D, CNN dual_sal and the ensemble with a fixed combiner, plus the OOF frame."""
    run = os.path.join(C2, d, ARM_SOURCES[arm], f"cf_{k}", "ensemble")
    cfg = json.load(open(os.path.join(run, "run_config.json")))
    assert (cfg["attr_model"], cfg["cnn_model"], cfg["attr_cf_count"], cfg["cnn_cf_count"]) == \
        ("MLP_I+C+D", "dual_sal", k, k), (d, arm, k, cfg)
    oof = pd.read_csv(os.path.join(run, "oof_predictions.csv"))
    auc = lambda col: np.array([roc_auc_score(g["correct"], g[col]) for _, g in oof.groupby("fold")])
    return {"attr": auc("MLP_I+C+D_prob"), "cnn": auc("dual_sal_prob"), "ens": auc(f"ens_{combiner}")}, oof

def k_arm_table():
    """One row per (disease, arm). Returns (numeric frame, formatted rows for LaTeX)."""
    k1, k3 = KARM_K
    long = []
    for d in DISEASES:
        ref = None
        for arm in ["KNN", "diffusion"]:
            _, combiner, _ = arm_fold_aucs(d, arm, k1)      # the arm's best combiner at K=1 (Section 5)
            (f1, o1), (f3, o3) = karm_fold_aucs(d, arm, k1, combiner), karm_fold_aucs(d, arm, k3, combiner)
            # Paired across K and across arms: same rows in the same folds everywhere.
            rows_k1 = set(zip(o1["path"], o1["fold"]))
            assert rows_k1 == set(zip(o3["path"], o3["fold"])), f"{d} {arm}: K={k1} and K={k3} rows differ"
            ref = ref or rows_k1
            assert rows_k1 == ref, f"{d}: arms differ"
            num = {"disease": d, "arm": arm, "combiner": combiner}
            for fam in ["attr", "cnn", "ens"]:
                a, b = f1[fam].mean(), f3[fam].mean()
                bt = KARM_BOOT.loc[(d, arm, fam)]
                # The bootstrap must have tested this delta: same models, same combiner, same rows.
                # 1e-5: the bootstrap breaks the few tied scores by sort order, sklearn averages them.
                assert abs(bt.observed - (b - a)) < 1e-5, (d, arm, fam, bt.observed, b - a)
                assert fam != "ens" or bt.combiner == combiner, (d, arm, bt.combiner, combiner)
                num.update({f"{fam} K{k1}": a, f"{fam} K{k3}": b, f"{fam} delta": b - a,
                            f"{fam} p_adj": bt.p_adj})
            long.append(num)
    tab = pd.DataFrame(long).set_index(["disease", "arm"])
    mean = tab.drop(columns="combiner").groupby(level="arm", sort=False).mean()
    for arm in mean.index:
        for fam in ["attr", "cnn", "ens"]:
            bt = KARM_BOOT.loc[("AVERAGE", arm, fam)]
            assert abs(bt.observed - mean.loc[arm, f"{fam} delta"]) < 1e-5, (arm, fam)
            mean.loc[arm, f"{fam} p_adj"] = bt.p_adj
    mean.index = pd.MultiIndex.from_product([["Mean"], mean.index], names=tab.index.names)
    tab = pd.concat([tab, mean])

    rows = []
    for (d, arm), r in tab.iterrows():
        if d == "Mean" and not KARM_SHOW_MEAN:
            continue
        row = {"Disease": "\\textit{Mean}" if d == "Mean" else DISEASE_NAMES[d],
               "Arm": "KNN" if arm == "KNN" else "Diff."}
        for fam in ["attr", "cnn", "ens"]:
            row[f"{fam}_1"], row[f"{fam}_3"] = f"${r[f'{fam} K{k1}']:.3f}$", f"${r[f'{fam} K{k3}']:.3f}$"
            sig = stars(r[f"{fam} p_adj"])
            dv = f"{r[f'{fam} delta']:+.4f}"
            row[f"{fam}_d"] = f"$\\mathbf{{{dv}}}^{{{sig}}}$" if sig else f"${dv}$"   # bold = significant
        rows.append(row)
    return tab, rows

karm_tab, karm_rows = k_arm_table()
karm_tab.round(4)


In [ ]:
k1, k3 = KARM_K
fams = [("attr", "Attribute model (I+Cf+D)"), ("cnn", "CNN (Dual+S)"), ("ens", "Ensemble")]
fam_cols = [f"{fam}_{s}" for fam, _ in fams for s in ["1", "3", "d"]]
combiners = "; ".join(f"{DISEASE_NAMES[d]} " + "/".join(COMBINER[karm_tab.loc[(d, a), "combiner"]]
                                                        for a in ["KNN", "diffusion"]) for d in DISEASES)
tex = latex_table(
    karm_rows, ["Disease", "Arm", *fam_cols], group_col="Disease",
    header=["Disease", "Arm", *[f"$K{{=}}{k1}$", f"$K{{=}}{k3}$", "$\\Delta$"] * len(fams)],
    col_fmt="ll|" + "|".join(["ccr"] * len(fams)),
    pre_header=["\\multicolumn{2}{c|}{} & " + " & ".join(
                    f"\\multicolumn{{3}}{{c{'|' if i < len(fams) - 1 else ''}}}{{{h}}}" for i, (_, h) in enumerate(fams))
                + " \\\\",
                " ".join(f"\\cmidrule({'lr' if i < len(fams) - 1 else 'l'}){{{3 + 3 * i}-{5 + 3 * i}}}"
                         for i in range(len(fams)))],
    caption=f"Effect of the number of counterfactuals, $K={k1}$ versus $K={k3}$, for retrieved (KNN) and diffusion "
            f"counterfactuals, on the same 20{{,}}000-image subset, folds and seed. AUROC for predicting C0 "
            f"correctness, mean over 5 CV folds.",
    label="tab:k_arm_sub20k",
    notes="KNN: $K$ nearest retrieved neighbours with the opposite C0 prediction. Diff.: $K$ diffusion edits of the "
          "query at decreasing strength ($K=1$: high; $K=3$: high, mid and low). $\\Delta$: $K=3$ minus $K=1$. "
          "Ensemble: the attribute model and the CNN combined with the arm's best combiner at $K=1$, kept fixed at "
          f"$K=3$ (KNN/diffusion: {combiners}). Stars: paired patient-level bootstrap of $\\Delta$ "
          f"({BOOT_META['n_boot']} replicates), Holm-adjusted across the five diseases within each arm and model"
          + ("; for the Mean rows, the bootstrap of the five-disease average, a single test" if KARM_SHOW_MEAN else "")
          + f"; {STARS_KEY}; no star: not significant. Bold: significant $\\Delta$."
          + (" Mean: average over the five diseases." if KARM_SHOW_MEAN else ""),
)
save_table(tex, "k_arm_sub20k")


## 5c. Diffusion only, K = 1 vs K = 3, 20k subset

Section 5b without the KNN rows: the diffusion arm alone, one row per disease. K=1 is the `high` edit; K=3 pools
`high`, `mid` and `low`. Same models, numbers and bootstrap stars as 5b (the diffusion rows of `karm_tab`; the Holm
families are within the arm, so dropping KNN changes no star). **Δ** is K=3 − K=1, bold when significant.

In [ ]:
k1, k3 = KARM_K
DIFF_K_SHOW_MEAN = True
dk = karm_tab.xs("diffusion", level="arm")
dk_rows = []
for d, r in dk.iterrows():
    if d == "Mean" and not DIFF_K_SHOW_MEAN:
        continue
    row = {"Disease": "\\textit{Mean}" if d == "Mean" else DISEASE_NAMES[d]}
    for fam in ["attr", "cnn", "ens"]:
        row[f"{fam}_1"], row[f"{fam}_3"] = f"${r[f'{fam} K{k1}']:.3f}$", f"${r[f'{fam} K{k3}']:.3f}$"
        sig, dv = stars(r[f"{fam} p_adj"]), f"{r[f'{fam} delta']:+.4f}"
        row[f"{fam}_d"] = f"$\\mathbf{{{dv}}}^{{{sig}}}$" if sig else f"${dv}$"   # bold = significant
    dk_rows.append(row)

fams = [("attr", "Attribute model (I+Cf+D)"), ("cnn", "CNN (Dual+S)"), ("ens", "Ensemble")]
diff_combiners = "; ".join(f"{DISEASE_NAMES[d]} {COMBINER[dk.loc[d, 'combiner']]}" for d in DISEASES)
tex = latex_table(
    dk_rows, ["Disease", *[f"{fam}_{s}" for fam, _ in fams for s in ["1", "3", "d"]]], group_col="Disease",
    group_rules=False,
    header=["Disease", *[f"$K{{=}}{k1}$", f"$K{{=}}{k3}$", "$\\Delta$"] * len(fams)],
    col_fmt="l|" + "|".join(["ccr"] * len(fams)),
    pre_header=[" & " + " & ".join(
                    f"\\multicolumn{{3}}{{c{'|' if i < len(fams) - 1 else ''}}}{{{h}}}" for i, (_, h) in enumerate(fams))
                + " \\\\",
                " ".join(f"\\cmidrule({'lr' if i < len(fams) - 1 else 'l'}){{{2 + 3 * i}-{4 + 3 * i}}}"
                         for i in range(len(fams)))],
    caption=f"Effect of the number of diffusion counterfactuals, $K={k1}$ versus $K={k3}$, on the 20{{,}}000-image "
            f"subset. AUROC for predicting C0 correctness, mean over 5 CV folds.",
    label="tab:k_diffusion_sub20k",
    notes="$K=1$: the strongest (high) diffusion edit of the query; $K=3$: the high, mid and low edits pooled. "
          "$\\Delta$: $K=3$ minus $K=1$. Ensemble: the attribute model and the CNN combined with the best combiner "
          f"at $K=1$, kept fixed at $K=3$ ({diff_combiners}). Stars: paired patient-level bootstrap of $\\Delta$ "
          f"({BOOT_META['n_boot']} replicates), Holm-adjusted across the five diseases within each model"
          + ("; for the Mean row, the bootstrap of the five-disease average, a single test"
             if DIFF_K_SHOW_MEAN else "")
          + f"; {STARS_KEY}; no star: not significant. Bold: significant $\\Delta$."
          + (" Mean: average over the five diseases." if DIFF_K_SHOW_MEAN else ""),
).replace("\n\\textit{Mean} &", "\n\\midrule\n\\textit{Mean} &")
save_table(tex, "k_diffusion_sub20k")


## 5d. How much data KNN needs to match one diffusion counterfactual

The dataset-size sweep of `SubsetsC2.ipynb` (`code/c2_knn_size_sweep.py`, results in
`C2_final_results/<disease>/knn_size_sweep/sweep_pooled.csv`). Attribute MLP **I+C+D**, probability-free.

* **Test**: the 5 folds of the 20k diffusion subset. AUROC is pooled over the out-of-fold predictions, so every image
  is scored once, by the model whose fold held it out; folds are identical in every cell.
* **KNN**: trained on N rows drawn from the whole C2 set minus the test fold's patients (nested, patient-whole
  subsamples, 5 seeds). The N rows are also the only rows counterfactuals are retrieved from.
* **Diffusion target**: K=1 (`high` edit), trained on the full 16k train side of the subset, mean over the same 5 seeds.

Each cell is the smallest N (from 1k, 2k, 4k, 8k, 16k, 32k and the full pool) at which KNN at that K, mean over seeds,
reaches the diffusion target.

In [ ]:
NSTAR_CONFIG = "I+C+D"
NSTAR_K      = [1, 3, 5, 10, 20]

def nstar_table():
    """One row per disease: diffusion AUROC at 16k and the smallest N where KNN at each K reaches it."""
    long, rows = [], []
    for d in DISEASES:
        r = pd.read_csv(os.path.join(C2, d, "knn_size_sweep", "sweep_pooled.csv"))
        r = r[r["config"] == NSTAR_CONFIG]
        target = r[(r["arm"] == "diffusion") & (r["n_target"] == "20k")]["auc"].mean()
        knn = (r[r["arm"] == "knn"].groupby(["k", "n_target"])
                 .agg(auc=("auc", "mean"), n=("n_train_mean", "mean")).reset_index())
        row, num = {"Disease": DISEASE_NAMES[d], "dif": f"${target:.3f}$"}, {"disease": d, "diffusion": target}
        for k in NSTAR_K:
            s = knn[knn["k"] == k].sort_values("n")
            hit = s[s["auc"] >= target]
            if not len(hit):
                row[f"K{k}"], num[f"K{k}"] = "$>$all", np.inf
            else:
                h = hit.iloc[0]
                label = f"{h.n / 1000:.0f}k"
                row[f"K{k}"] = f"all ({label})" if h.n_target == "all" else label
                num[f"K{k}"] = h.n
        rows.append(row)
        long.append(num)
    return pd.DataFrame(long).set_index("disease"), rows

nstar_tab, nstar_rows = nstar_table()
nstar_tab.round(4)

In [ ]:
k_cols = [f"K{k}" for k in NSTAR_K]
tex = latex_table(
    nstar_rows, ["Disease", "dif", *k_cols], group_col="Disease", group_rules=False,
    header=["Disease", "Diffusion AUROC", *[str(k) for k in NSTAR_K]],
    col_fmt="lc|" + "c" * len(NSTAR_K),
    pre_header=["& & \\multicolumn{%d}{c}{KNN training rows needed at $K =$} \\\\" % len(NSTAR_K),
                "\\cmidrule(l){3-%d}" % (2 + len(NSTAR_K))],
    caption="Training rows KNN needs to match one diffusion counterfactual. Diffusion AUROC: the attribute model "
            "(I+Cf+D, probability-free) with one diffusion counterfactual ($K=1$), trained on the full 16k train "
            "side of the 20{,}000-image subset. Each KNN cell is the smallest training-set size $N$ (from 1k, 2k, 4k, "
            "8k, 16k, 32k and the full pool, ``all'') at which the same model with $K$ retrieved neighbours reaches "
            "that AUROC. KNN draws its $N$ rows from the whole C2 set minus the test fold's patients and retrieves "
            "only from them. AUROC is computed out-of-fold over the 20{,}000-image subset (5-fold CV: each image is "
            "tested once, by the model whose fold held it out), with identical folds in every cell; both arms are "
            "averaged over 5 seeds.",
    label="tab:subsets_knn_nstar",
)
save_table(tex, "subsets_knn_nstar")

## 6. Trajectory models, diffusion counterfactuals, 20k subset, K = 1

The trajectory models of Section 1e of `C2_evaluation_diffusion.ipynb` (figure `diffusion_aim1e_trajectory`), written by
`c2_train_trajectory.py` to `C2_final_results/<disease>/diffusion_hml_sub20k/cf_1/trajectory/`. They see neither the
counterfactual image nor C0's probability on the query, only C0's probability along the FastDiME denoising trajectory
(every 5 steps, 30 values). Trained on the `high` level only; K=1 is that level.

* **MLP** on each block of the 32 hand-made trajectory features, then on all of them (**ALL**): `LEVEL` (where the
  trajectory sits, whether it crosses the threshold), `MONO` (how steadily it moves), `SHAPE` (slope, curvature, cubic fit),
  `DYN` (path length, velocity, acceleration).
* **1D CNN** and **bidirectional GRU** on the whole sequence (logit of the target-class probability + its step change).

Cells are mean ± std AUROC over the 5 folds, recomputed from the fold predictions. Bold marks the best model in each row.
No MSP column.

In [ ]:
TRAJ_SOURCE = "diffusion_hml_sub20k"
TRAJ_K      = 1
TRAJ_MODELS = {"LEVEL": "MLP_LEVEL_prob", "MONO": "MLP_MONO_prob", "SHAPE": "MLP_SHAPE_prob",
               "DYN": "MLP_DYN_prob", "ALL": "MLP_ALL_prob", "CNN": "CNN_TRAJ_prob", "RNN": "RNN_TRAJ_prob"}
TRAJ_LABELS = {"LEVEL": "LEVEL", "MONO": "MONO", "SHAPE": "SHAPE", "DYN": "DYN", "ALL": "ALL",
               "CNN": "1D CNN", "RNN": "BiGRU"}
TRAJ_MLP    = ["LEVEL", "MONO", "SHAPE", "DYN", "ALL"]

def traj_dir(disease, k=TRAJ_K, source=TRAJ_SOURCE):
    return os.path.join(C2, disease, source, f"cf_{k}", "trajectory")

def traj_table():
    """One row per disease. Returns (numeric frame of mean-fold AUROCs, formatted rows for LaTeX)."""
    long, rows = [], []
    for d in DISEASES:
        folds = fold_predictions(traj_dir(d))
        aucs  = {m: np.array([roc_auc_score(f["correct"], f[col]) for f in folds]) for m, col in TRAJ_MODELS.items()}
        # Recomputed folds must reproduce the script's own summary.
        s = pd.read_csv(os.path.join(traj_dir(d), "cv_summary.csv"))
        for r in s.itertuples():
            m = r.model if r.config == "TRAJ" else r.config
            if m in aucs:
                assert np.isclose(aucs[m].mean(), r.auc_mean, atol=1e-4), (d, m, aucs[m].mean(), r.auc_mean)
        best = max(aucs, key=lambda m: aucs[m].mean())
        rows.append({"Disease": DISEASE_NAMES[d], **{m: fmt_cell(f, bold=(m == best)) for m, f in aucs.items()}})
        long.append({"disease": d, "n": sum(len(f) for f in folds), **{m: f.mean() for m, f in aucs.items()}})
    return pd.DataFrame(long).set_index("disease"), rows

traj_tab, traj_rows = traj_table()
traj_tab.round(4)

In [ ]:
tex = latex_table(
    traj_rows, ["Disease", *TRAJ_MODELS], group_col="Disease", group_rules=False,
    header=["Disease", *[TRAJ_LABELS[m] for m in TRAJ_MODELS]],
    col_fmt="l" + "c" * len(TRAJ_MLP) + "|cc",
    pre_header=["& \\multicolumn{%d}{c|}{MLP on trajectory features} & \\multicolumn{2}{c}{Sequence models} \\\\"
                % len(TRAJ_MLP),
                "\\cmidrule(lr){2-%d} \\cmidrule(l){%d-%d}" % (1 + len(TRAJ_MLP), 2 + len(TRAJ_MLP), 3 + len(TRAJ_MLP))],
    caption=f"AUROC for predicting C0 correctness with the trajectory models, using diffusion counterfactuals at "
            f"$K={TRAJ_K}$ on the 20{{,}}000-image subset. The models see neither the counterfactual image nor C0's "
            f"probability on the query, only C0's probability along the denoising trajectory toward the counterfactual. "
            f"MLP: one model per block of the 32 hand-made trajectory features (LEVEL: where the trajectory sits and "
            f"whether it crosses C0's threshold; MONO: how steadily it moves; SHAPE: slope, curvature and cubic fit; "
            f"DYN: path length, velocity and acceleration) and on all of them (ALL). Sequence models read the whole "
            f"trajectory. Values are mean $\\pm$ std over 5 CV folds; bold is the best model per disease.",
    label="tab:traj_diffusion",
    resize=True,
)
save_table(tex, "traj_diffusion")

## 7. Prior-art baselines (Aim 2), KNN counterfactuals, K = 10

The final system against three published failure predictors, all on the ensemble's folds and test rows (asserted
by their scripts). ConfidNet and TrustScore see **no counterfactual** and read C0's frozen penultimate features; SCCE
reads only the counterfactual's **distance**:

* **ConfidNet** (Corbière et al. 2019, `code/c2_confidnet.py`): an MLP head trained to regress the threshold-adapted
  true-class probability.
* **TrustScore** (Jiang et al. 2018, `code/c2_trustscore.py`): distance to the nearest training points of the other
  class ÷ distance to those of the predicted class, after density filtering; PCA dims and α chosen on a patient-grouped
  inner split.
* **SCCE** (Bonsignori et al. 2026, `code/c2_scce.py`): L1 distance to the nearest of the ensemble's own K=10 KNN
  counterfactuals, in the retrieval's standardised attribute space (our counterfactuals instead of Growing Spheres).
* **Ours**: the final system, the ensemble of the attribute model and the CNN (the bootstrap's `best_all` pick).

**Table 7a** is the main table: our method against the adjusted MSP and the three baselines, one block per metric
(AUROC ↑, E-AURC ↓), a row per disease and their mean. Cells are mean ± std over the 5 folds, recomputed here from the
out-of-fold predictions (`c2_scce_compare.build_table` checks rows, folds, labels and patients across all models,
and every mean must reproduce the bootstrap summary). The Mean row's std is over folds of the five-disease average.
Bold is the best method per row. Stars on our method: its paired bootstrap test against the adjusted MSP (the 2a
test, Holm across the five diseases; Mean row: the average's test). Against ConfidNet, TrustScore and SCCE the
tests are in Table 7b; the notes state the outcome, and any baseline cell where ours is *not* significantly better
is marked.

**Table 7b** lists the paired tests: Δ (observed, positive = the left-hand model better; E-AURC as right − left) with stars
from the patient-level bootstrap replayed by `c2_confidnet_compare.py` / `c2_trustscore_compare.py` / `c2_scce_compare.py` (same draws as
`bootstrapping.ipynb`), Holm-adjusted across the five diseases within each comparison and metric. The Mean row is the
bootstrap of the five-disease average, a single test.


In [ ]:
import c2_scce_compare as SCC
import c2_boot_extra as BX

BASE_K       = BOOT_META["final"]["k"]
BASE_MODELS  = ["MSP", "ConfidNet", "TrustScore", "SCCE", "Ensemble"]      # Table 7a, left to right
BASE_LABELS  = {"MSP": MSP_LABEL, "ConfidNet": "ConfidNet", "TrustScore": "TrustScore", "SCCE": "SCCE",
                "Ensemble": "Ours"}
BASE_METRICS = {"AUROC": (max, 3), "E-AURC": (min, 4)}        # (better, decimals)
BASE_METRIC_LABEL = {"AUROC": "AUROC $\\uparrow$", "E-AURC": "E-AURC $\\downarrow$"}
BASE_SHOW_MEAN = True
BASE_CMPS = {"ConfidNet - MSP": "confidnet_vs_msp", "TrustScore - MSP": "trustscore_vs_msp",
             "SCCE - MSP": "scce_vs_msp",
             "Ensemble - ConfidNet": "ensemble_vs_confidnet",
             "Ensemble - TrustScore": "ensemble_vs_trustscore", "Ensemble - SCCE": "ensemble_vs_scce"}
assert BOOT_META["msp_variant"] == MSP_VARIANT and not BOOT_META["final"]["prob"]

BASE_SUMMARY = pd.read_csv(os.path.join(C2, "bootstrap_baselines_summary.csv")).set_index(["disease", "method"])
BASE_TESTS   = pd.concat([pd.read_csv(os.path.join(C2, f"bootstrap_{f}.csv")) for f in BASE_CMPS.values()],
                         ignore_index=True)
assert set(BASE_TESTS.comparison) == set(BASE_CMPS), sorted(set(BASE_TESTS.comparison))
# Ours vs MSP: the 2a tests -- AUROC from the main export, E-AURC from its E-AURC section.
_EAURC = pd.read_csv(os.path.join(C2, "bootstrap_eaurc_deltas.csv"))


def ours_vs_msp(metric, d):
    """The bootstrap row of the final system against the adjusted MSP (figure 2a's test)."""
    if metric == "AUROC":
        r = BOOT_DELTAS[(BOOT_DELTAS.comparison == "best overall - MSP") & (BOOT_DELTAS.disease == d)]
    else:
        r = _EAURC[_EAURC.disease == d]
    assert len(r) == 1, (metric, d, len(r))
    return r.iloc[0]


def base_fold_scores(d):
    """{metric: {model: per-fold array}} on the rows and folds every model shares."""
    tab = SCC.build_table(d, BASE_K, BOOT_META["picks"][d]["best_all"], MSP_VARIANT)
    out = {"AUROC": {}, "E-AURC": {}}
    g = [f for _, f in tab.groupby("fold")]
    for m in BASE_MODELS:
        out["AUROC"][m]  = np.array([roc_auc_score(f["correct"], f[m]) for f in g])
        out["E-AURC"][m] = np.array([BX.e_aurc(f["correct"], f[m]) for f in g])
        for metric in out:
            # 1e-6: the bootstrap breaks the few tied scores by sort order, sklearn averages them.
            assert abs(out[metric][m].mean() - BASE_SUMMARY.loc[(d, m), metric]) < 1e-6, (d, m, metric)
    return out


def baseline_table():
    """Table 7a. Returns (numeric long frame, LaTeX rows, baseline cells where ours is NOT significantly
    better)."""
    folds = {d: base_fold_scores(d) for d in DISEASES}
    long, rows, not_beaten = [], [], []
    for metric, (better, dec) in BASE_METRICS.items():
        blocks = [(d, DISEASE_NAMES[d], {m: folds[d][metric][m] for m in BASE_MODELS}) for d in DISEASES]
        if BASE_SHOW_MEAN:
            # Fold-wise average over the five diseases: its mean is the mean of the disease means.
            blocks.append(("AVERAGE", "\\textit{Mean}",
                           {m: np.mean([folds[d][metric][m] for d in DISEASES], axis=0) for m in BASE_MODELS}))
        for d, name, sc in blocks:
            best = better(BASE_MODELS, key=lambda m: sc[m].mean())
            row = {"Metric": BASE_METRIC_LABEL[metric], "Disease": name}
            for m in BASE_MODELS:
                cell = fmt_cell(sc[m], bold=(m == best), digits=dec)
                if m == "Ensemble":
                    sig = stars(ours_vs_msp(metric, d).p_adj)
                    cell = cell[:-1] + f"^{{{sig}}}$" if sig else cell
                elif m != "MSP" and base_test(f"Ensemble - {m}", metric, d).p_adj >= BOOT_META["alpha"]:
                    cell = cell[:-1] + "^{\\dagger}$"
                    not_beaten.append((metric, d, m))
                row[m] = cell
                long.append({"metric": metric, "disease": d, "model": m, "mean": sc[m].mean(),
                             "std": sc[m].std()})
            rows.append(row)
    return pd.DataFrame(long), rows, not_beaten


def base_test(cmp, metric, d):
    r = BASE_TESTS[(BASE_TESTS.comparison == cmp) & (BASE_TESTS.metric == metric) & (BASE_TESTS.disease == d)]
    assert len(r) == 1, (cmp, metric, d, len(r))
    return r.iloc[0]


def baseline_tests_table():
    long, rows = [], []
    for d in DISEASES + (["AVERAGE"] if BASE_SHOW_MEAN else []):
        for metric, (_, dec) in BASE_METRICS.items():
            row = {"Disease": "Mean" if d == "AVERAGE" else DISEASE_NAMES[d], "Metric": metric}
            for c in BASE_CMPS:
                r = base_test(c, metric, d)
                sig = stars(r.p_adj)
                v = f"{r.gain:+.{dec}f}"
                row[c] = f"${v}^{{{sig}}}$" if sig else f"${v}$"
                long.append({"disease": d, "metric": metric, "comparison": c, "delta": r.gain,
                             "boot lo": r.lo, "boot hi": r.hi, "p_adj": r.p_adj})
            rows.append(row)
    return pd.DataFrame(long), rows


base_long, base_rows, BASE_NOT_BEATEN = baseline_table()
test_long, test_rows = baseline_tests_table()
display(base_long.assign(cell=base_long["mean"].round(4).astype(str) + " ± " + base_long["std"].round(4).astype(str))
        .pivot(index=["metric", "disease"], columns="model", values="cell")[BASE_MODELS])
print("baseline cells where ours is NOT significantly better:", BASE_NOT_BEATEN or "none")
test_long.pivot_table(index=["metric", "disease"], columns="comparison", values=["delta", "p_adj"])[
    [(v, c) for v in ("delta", "p_adj") for c in BASE_CMPS]].round(4)


In [ ]:
beaten_note = ("Ours is significantly better than ConfidNet, TrustScore and SCCE in every row "
               "(paired bootstrap, Holm-adjusted, Table~\\ref{tab:baselines_tests_knn})." if not BASE_NOT_BEATEN else
               "$\\dagger$: ours is not significantly better than this baseline (paired bootstrap, Holm-adjusted, "
               "Table~\\ref{tab:baselines_tests_knn}); everywhere else it is.")
tex = latex_table(
    base_rows, ["Metric", "Disease", *BASE_MODELS], group_col="Metric",
    header=["", "Disease", *[BASE_LABELS[m] for m in BASE_MODELS]],
    col_fmt="ll|c|ccc|c",
    caption=f"Our method against the adjusted MSP and three prior-art failure predictors at $K={BASE_K}$ (KNN "
            f"counterfactuals), for predicting C0 correctness. Mean $\\pm$ std over 5 CV folds.",
    label="tab:baselines_knn",
    notes="ConfidNet: MLP on C0's frozen penultimate features trained to regress the threshold-adapted true-class "
          "probability. TrustScore: distance to the nearest training points of the other class divided by that to the "
          "predicted class, after density filtering. SCCE: L1 distance to the nearest of our own counterfactuals in "
          "the retrieval's standardised attribute space. Ours: the ensemble of the attribute model and the CNN. "
          "Bold: the best method per row. Stars on ours: paired patient-level bootstrap against the adjusted MSP "
          f"({BOOT_META['n_boot']} replicates), Holm-adjusted across the five diseases; for the Mean row the test of "
          f"the five-disease average; {STARS_KEY}; no star: not significant. " + beaten_note
          + (" Mean: average over the five diseases; its std is over folds of the fold-wise five-disease average."
             if BASE_SHOW_MEAN else ""),
)
# Set the Mean row apart: a light rule across the data columns above it in each block.
tex = "\n".join("\\cmidrule(l){2-%d}\n%s" % (2 + len(BASE_MODELS), l)
                if l.lstrip().startswith("& \\textit{Mean}") else l for l in tex.split("\n"))
save_table(tex, "baselines_knn")

tex = latex_table(
    test_rows, ["Disease", "Metric", *BASE_CMPS], group_col="Disease",
    header=["Disease", "", "ConfidNet", "TrustScore", "SCCE", "ConfidNet", "TrustScore", "SCCE"],
    col_fmt="ll|rrr|rrr",
    pre_header=["\\multicolumn{2}{c|}{} & \\multicolumn{3}{c|}{Baseline $-$ " + MSP_LABEL + "} "
                "& \\multicolumn{3}{c}{Ensemble $-$ baseline} \\\\",
                "\\cmidrule(lr){3-5} \\cmidrule(l){6-8}"],
    caption=f"Paired comparisons of the prior-art baselines at $K={BASE_K}$ (KNN counterfactuals): each baseline "
            f"against the adjusted MSP, and the final system's ensemble against each baseline.",
    label="tab:baselines_tests_knn",
    notes="$\\Delta$ of the mean-fold metric, positive = the left-hand model is better (for E-AURC, right minus left). "
          f"Stars: paired patient-level bootstrap ({BOOT_META['n_boot']} replicates, the draws of the main analysis), "
          f"Holm-adjusted across the five diseases within each comparison and metric"
          + ("; the Mean row is the bootstrap of the five-disease average, a single test" if BASE_SHOW_MEAN else "")
          + f"; {STARS_KEY}; no star: not significant.",
)
save_table(tex, "baselines_tests_knn")


## 8. Diffusion soft targets, per task

The probabilities the diffusion counterfactuals are steered to (`CF_levels/<disease>/levels.json`, computed by `compute_levels` in `code/cf_generate_levels.py`). Per disease and direction of travel there are three equally spaced targets, **low**, **mid** and **high**: the 5th and 95th percentile of the target class's own correctly classified cases, and their midpoint. Each disease has its own targets because C0's probability scale differs between diseases.

In [ ]:
LV_DIR = os.path.join(os.environ.get("THESIS_RESULTS", "/work3/s251710/thesis_results"), "CF_levels")
LV_NAMES = ["low", "mid", "high"]

lv_rows = []
for d in DISEASES:
    lv = json.load(open(os.path.join(LV_DIR, d, "levels.json")))
    assert lv["level_names"] == LV_NAMES
    row = {"Disease": DISEASE_NAMES[d], "thr": f"${lv['threshold']:.3f}$"}
    for direction, key in (("neg", "to_negative"), ("pos", "to_positive")):
        for n, v in zip(LV_NAMES, lv[key]):
            row[f"{direction}_{n}"] = f"${v:.3f}$"
    lv_rows.append(row)

lv_cols = ["Disease", "thr", *[f"{s}_{n}" for s in ("neg", "pos") for n in LV_NAMES]]
tex = latex_table(
    lv_rows, lv_cols, group_col="Disease", group_rules=False,
    header=["Disease", "Threshold", *LV_NAMES, *LV_NAMES],
    col_fmt="lc|ccc|ccc",
    pre_header=["& & \\multicolumn{3}{c|}{Towards negative} & \\multicolumn{3}{c}{Towards positive} \\\\",
                "\\cmidrule(lr){3-5} \\cmidrule(l){6-8}"],
    caption="Soft targets of the diffusion counterfactuals, per disease: the $C_0$ probability each of the three "
            "edits (low, mid, high) is steered to, in each direction of travel.",
    label="tab:diffusion_soft_targets",
    notes="Targets are $C_0$ probabilities. Towards positive: low, mid and high are the 5th percentile, midpoint "
          "and 95th percentile of the correctly classified positives. Towards negative: the mirror, the 95th "
          "percentile, midpoint and 5th percentile of the correctly classified negatives, so that high is always "
          "the edit deepest into the target class. Threshold: $C_0$'s operating threshold, which separates the "
          "two directions.",
)
save_table(tex, "diffusion_soft_targets")
pd.DataFrame(lv_rows).set_index("Disease")

## 9. Attributes that go with C0's errors, per side of C0's call

The step-4 selection of `attribute_explainability.ipynb`: per disease and per side of C0's call, the top `N_KEEP` = 5
non-redundant attributes by correlation with C0's correctness (KNN counterfactuals, nearest counterfactual, all five
folds pooled). Read from `figures/xai/<disease>/<disease>_knn_correct_cf_attribute_correlations.csv`, which that notebook
writes in step 4 (run it per disease with `XAI_DISEASE=<disease>`, at least down to step 4).

* **Rank**: the attribute's place in the full ranking of 461 attributes by |r| of its strongest form. Gaps show where a
  higher-ranked attribute was skipped as redundant (|r| ≥ 0.6 with one already kept).
* **r (I / Cf / D)**: point-biserial r with `correct` of the image's value, its counterfactual's and their difference.
  Bold = the strongest form, the one the attribute is ranked by. Positive = higher values go with correct calls.
* **d**: the strongest form's mean on correct minus wrong calls, in SD units of that side (same sign as r).
* **r(MSP)**: the strongest form's r with threshold MSP on the same calls. **r | MSP**: its partial r with
  correctness, MSP held fixed: what is left once C0's own confidence is known.

In [ ]:
import re

TOPATTR_SOURCE = "knn_correct_cf"
TOPATTR_FORMS  = ["image", "CF", "delta"]
TOPATTR_SIDES  = {1: "Positive", 0: "Negative"}
XAI_DIR        = os.path.abspath(os.path.join("figures", "xai"))
# Region names as in Table 0b; the C1 segmenter's own names otherwise.
REGION_NAMES = {"Facies Diaphragmatica": "Diaphragm", "Hilus Pulmonis": "hilum",
                "Weasand": "Trachea and oesophagus"}

def attr_label(a):
    """'Right Hilus Pulmonis_original_firstorder_Median' -> 'Right hilum: median'."""
    if a.split("_", 1)[0].islower():                              # clinical ratios, demographics
        return a.replace("_pred", " (predicted)").replace("_", " ").capitalize()
    region, rest = a.split("_", 1)
    for k, v in REGION_NAMES.items():
        region = region.replace(k, v)
    region = region[0].upper() + region[1:].lower()
    rest = rest.replace("original_firstorder_", "").replace("original_shape2D_", "")
    rest = rest.replace("area_pixels", "area").replace("_", " ")
    rest = re.sub(r"(\d+)Percentile", r"\1th percentile", rest)
    rest = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", rest).lower()
    return f"{region}: {rest}"

def top_attributes(d, source=TOPATTR_SOURCE):
    """The step-4 selection of attribute_explainability.ipynb for disease d, both sides, in selection order."""
    path = os.path.join(XAI_DIR, d, f"{d}_{source}_attribute_correlations.csv")
    assert os.path.exists(path), f"{path} missing: run attribute_explainability.ipynb with XAI_DISEASE={d}"
    t = pd.read_csv(path)
    return t[t["selected"].notna()].sort_values(["side", "selected"], ascending=[False, True])

def num(x, digits=3, bold=False):
    return f"$\\mathbf{{{x:.{digits}f}}}$" if bold else f"${x:.{digits}f}$"

topattr = pd.concat([top_attributes(d) for d in DISEASES], ignore_index=True)

ta_rows = []
for d in DISEASES:
    for p, side in TOPATTR_SIDES.items():
        t = topattr[(topattr["disease"] == d) & (topattr["side"] == p)]
        for i, (_, r) in enumerate(t.iterrows()):
            best = r["best form"]
            ta_rows.append({
                "Disease": DISEASE_NAMES[d],
                "call": (f"\\multirow{{{len(t)}}}{{*}}{{\\shortstack[l]{{{side}\\\\"
                         f"${100 * r['error_rate']:.1f}\\%$ wrong}}}}") if i == 0 else "",
                "attr": attr_label(r["attribute"]), "rank": f"${int(r['rank'])}$",
                **{f"r_{f}": num(r[f"r_{f}"], bold=f == best) for f in TOPATTR_FORMS},
                "d": num(r[f"d_{best}"], 2), "r_msp": num(r["r_msp"]), "r_given_msp": num(r["r_given_msp"]),
            })

ta_cols = ["Disease", "call", "attr", "rank", *[f"r_{f}" for f in TOPATTR_FORMS], "d", "r_msp", "r_given_msp"]
tex = latex_table(
    ta_rows, ta_cols, group_col="Disease",
    header=["Disease", "$C_0$'s call", "Attribute", "Rank", "I", "Cf", "D", "$d$", "MSP", "$\\mid$ MSP"],
    col_fmt="lllr|ccc|c|cc",
    pre_header=["& & & & \\multicolumn{3}{c|}{$r$ with correctness} & & \\multicolumn{2}{c}{$r$} \\\\",
                "\\cmidrule(lr){5-7} \\cmidrule(l){9-10}"],
    caption="The five non-redundant attributes most correlated with $C_0$'s correctness, per disease and per side "
            "of $C_0$'s call (KNN counterfactuals, nearest counterfactual, all folds pooled).",
    label="tab:top_attributes",
    notes="Within each side, attributes are ranked by $|r|$ of their strongest form, and an attribute is kept only if "
          "its $|r|$ with every attribute already kept is below $0.6$ (on the image values). Rank: place in the full "
          "ranking of all attributes; gaps are attributes skipped as redundant. $r$: point-biserial correlation with "
          "correctness of the image's value (I), its counterfactual's (Cf) and their difference (D); bold marks the "
          "strongest form, which the remaining columns use. Positive $r$: higher values go with correct calls. "
          "$d$: mean on correct minus wrong calls, in standard deviations of that side. $r$ MSP: correlation with "
          "threshold MSP on the same calls. $r \\mid$ MSP: partial correlation with correctness, MSP held fixed. "
          "Percentages: share of that side's calls $C_0$ gets wrong.",
)
# A lighter rule between the two sides of one disease, and a smaller font for eleven columns.
tex = tex.replace("\n & \\multirow{5}{*}{\\shortstack[l]{Negative", "\n\\cmidrule(l){2-10}\n & \\multirow{5}{*}{\\shortstack[l]{Negative")
tex = tex.replace("\\small", "\\footnotesize", 1)
save_table(tex, "top_attributes")

(topattr.assign(attribute=topattr["attribute"].map(attr_label))
 .set_index(["disease", "side", "selected"])
 [["attribute", "rank", "best form", "r_image", "r_CF", "r_delta", "r_msp", "r_given_msp", "error_rate"]].round(3))

## 9b. Regions where C0's saliency goes with its errors, per side of C0's call

Table 9 for the saliency arm: the step-4 selection of `saliency_explainability.ipynb`. Per disease and per side of
C0's call, the `N_KEEP` = 4 non-redundant regions whose Grad-CAM focus correlates most with C0's correctness, plus
`Outside` (the image outside the body) as the contrast every saliency figure shows (KNN counterfactuals, nearest
counterfactual, all five folds pooled). Read from
`figures/xai/<disease>/<disease>_knn_correct_cf_saliency_region_correlations.csv`, which that notebook writes right after
step 4 (run it per disease with `XAI_DISEASE=<disease>`, at least down to that cell). Only the diseases in `SAL_DISEASES`.

* **Focus**: share of the map in the region divided by the region's share of the image, as log10.
* **Rank**: the region's place among the candidate regions by |r| of its stronger X-ray map. Whole lungs and `Outside`
  are not candidates (no rank). Gaps show where a higher-ranked region was skipped as redundant (|r| ≥ 0.6 of its
  support focus with one already kept).
* **r (X-ray / Cf, Sup. / Ag.)**: point-biserial r with `correct` of the focus of the map supporting the call and of the
  one against it, for the X-ray (C0's call) and its counterfactual (the counterfactual's own call). Bold = the stronger
  X-ray map, the one the region is ranked by. Positive = more focus there goes with correct calls.
* **d**, **r(MSP)**, **r | MSP**: as in Table 9, for the bold form.

In [ ]:
SAL_DISEASES = ["cardiomegaly", "effusion"]   # the diseases saliency_explainability.ipynb was run for
SAL_FORMS    = ["support", "against", "CF support", "CF against"]

def region_label(r):
    """Saliency region names as in Table 0b: 'Heart-Left Lung overlap' -> 'Heart--left lung overlap'."""
    r = r.replace("Facies Diaphragmatica", "Diaphragm").replace("-", "--")
    return r[0] + r[1:].lower()

def top_regions(d, source=TOPATTR_SOURCE):
    """The step-4 selection of saliency_explainability.ipynb for disease d (key regions, then Outside), both sides."""
    path = os.path.join(XAI_DIR, d, f"{d}_{source}_saliency_region_correlations.csv")
    assert os.path.exists(path), f"{path} missing: run saliency_explainability.ipynb with XAI_DISEASE={d}"
    t = pd.read_csv(path)
    return t[t["selected"].notna()].sort_values(["side", "selected"], ascending=[False, True])

topreg = pd.concat([top_regions(d) for d in SAL_DISEASES], ignore_index=True)

tr_rows = []
for d in SAL_DISEASES:
    for p, side in TOPATTR_SIDES.items():
        t = topreg[(topreg["disease"] == d) & (topreg["side"] == p)]
        for i, (_, r) in enumerate(t.iterrows()):
            best = r["best form"]
            tr_rows.append({
                "Disease": DISEASE_NAMES[d],
                "call": (f"\\multirow{{{len(t)}}}{{*}}{{\\shortstack[l]{{{side}\\\\"
                         f"${100 * r['error_rate']:.1f}\\%$ wrong}}}}") if i == 0 else "",
                "region": region_label(r["region"]),
                "rank": f"${int(r['rank'])}$" if pd.notna(r["rank"]) else "--",
                **{f"r_{f}": num(r[f"r_{f}"], bold=f == best) for f in SAL_FORMS},
                "d": num(r[f"d_{best}"], 2), "r_msp": num(r["r_msp"]), "r_given_msp": num(r["r_given_msp"]),
            })

tr_cols = ["Disease", "call", "region", "rank", *[f"r_{f}" for f in SAL_FORMS], "d", "r_msp", "r_given_msp"]
tex = latex_table(
    tr_rows, tr_cols, group_col="Disease",
    header=["Disease", "$C_0$'s call", "Region", "Rank", "Sup.", "Ag.", "Sup.", "Ag.", "$d$", "MSP", "$\\mid$ MSP"],
    col_fmt="lllr|cc|cc|c|cc",
    pre_header=["& & & & \\multicolumn{4}{c|}{$r$ with correctness} & & \\multicolumn{2}{c}{$r$} \\\\",
                "\\cmidrule(lr){5-8} \\cmidrule(l){10-11}",
                "& & & & \\multicolumn{2}{c|}{X-ray} & \\multicolumn{2}{c|}{Cf} & & & \\\\",
                "\\cmidrule(lr){5-6} \\cmidrule(lr){7-8}"],
    caption="The four non-redundant regions whose Grad-CAM focus is most correlated with $C_0$'s correctness, per "
            "disease and side of $C_0$'s call, and the image outside the body for contrast (KNN counterfactuals, "
            "nearest counterfactual, all folds pooled).",
    label="tab:top_regions",
    notes="Focus: share of the Grad-CAM in the region divided by the region's share of the image, as $\\log_{10}$. "
          "Sup.: the map supporting the call ($C_0$'s call for the X-ray, the counterfactual's own call for Cf); "
          "Ag.: the map against it. Within each side, regions are ranked by $|r|$ of the stronger X-ray map, and a "
          "region is kept only if the $|r|$ of its supporting focus with every region already kept is below $0.6$; "
          "whole lungs and Outside are not ranked. Rank: place among the ranked regions; gaps are regions skipped "
          "as redundant. $r$: point-biserial correlation with correctness; bold marks the stronger X-ray map, which "
          "the remaining columns use. Positive $r$: more focus in the region goes with correct calls. $d$: mean on "
          "correct minus wrong calls, in standard deviations of that side. $r$ MSP: correlation with threshold MSP "
          "on the same calls. $r \\mid$ MSP: partial correlation with correctness, MSP held fixed. Percentages: "
          "share of that side's calls $C_0$ gets wrong.",
)
# A lighter rule between the two sides of one disease, and a smaller font for eleven columns.
tex = re.sub(r"\n & (\\multirow\{\d+\}\{\*\}\{\\shortstack\[l\]\{Negative)",
             lambda m: f"\n\\cmidrule(l){{2-{len(tr_cols)}}}\n & {m.group(1)}", tex)
tex = tex.replace("\\small", "\\footnotesize", 1)
save_table(tex, "top_regions")

(topreg.assign(region=topreg["region"].map(region_label))
 .set_index(["disease", "side", "selected"])
 [["region", "rank", "best form", *[f"r_{f}" for f in SAL_FORMS], "r_msp", "r_given_msp", "error_rate"]].round(3))

## 10. C0 architecture generalisation, KNN counterfactuals, K = 10

Does the final system transfer to other C0 backbones? `code/c2_arch_compare.py` reran it with its recipe frozen
(KNN arm, K=10, attribute MLP **I+C+D**, CNN **Dual+S**, and the ensemble combiner DenseNet's bootstrap picked per
disease) on top of three C0s: DenseNet121 (the main C0), ViT-B/16 and ResNet-50. Nothing was re-tuned per C0.

One row per architecture and disease. **C0** columns: AUROC of C0 on its disease and its accuracy at its own Youden
threshold, on the held-out C2 rows. **C2** columns: AUROC for predicting C0 correctness of the adjusted MSP (at that
C0's own threshold), the attribute model, the CNN and ours (the ensemble). They are mean-fold AUROCs over the 5 CV folds
(`summary.csv` keeps bootstrap intervals, not a per-fold std, so cells carry no ±). **Δ** is ours − MSP.

Stars come from the script's single patient-level bootstrap (same patients resampled across every arch and disease):
each model against MSP (stars: significantly better; † on a model cell: significantly worse), Holm-adjusted across the five diseases within each (arch, comparison). The Mean row
averages the five diseases; its stars test the five-disease average, averaged inside each replicate.
Pre-registered criterion: the method generalises to a C0 if Ours − MSP is positive and significant on the diseases
where it holds for DenseNet.

In [ ]:
ARCH_DIR    = os.path.join(RESULTS, "C2_arch_compare")
ARCH_CFG    = json.load(open(os.path.join(ARCH_DIR, "run_config.json")))
ARCH_ORDER  = ["densenet121", "vit_b_16", "resnet50"]
ARCH_LABELS = {"densenet121": "DenseNet121", "vit_b_16": "ViT-B/16", "resnet50": "ResNet-50"}
ARCH_MODELS = ["MSP", "Attr", "CNN", "Ensemble"]
ARCH_SHOW_MEAN = True
assert ARCH_CFG["msp_variant"] == MSP_VARIANT, ARCH_CFG["msp_variant"]
assert set(ARCH_CFG["archs"]) == set(ARCH_ORDER) and set(ARCH_CFG["diseases"]) == set(DISEASES), ARCH_CFG

ARCH_SUMMARY = pd.read_csv(os.path.join(ARCH_DIR, "summary.csv")).set_index(["arch", "disease"])
ARCH_DELTAS  = pd.read_csv(os.path.join(ARCH_DIR, "deltas.csv"))

def arch_test(arch, model, d):
    """The bootstrap row of `model` - MSP (AUROC) for one arch and disease ("AVERAGE" for the mean)."""
    r = ARCH_DELTAS[(ARCH_DELTAS.arch == arch) & (ARCH_DELTAS.comparison == f"{model} - MSP")
                    & (ARCH_DELTAS.metric == "AUROC") & (ARCH_DELTAS.disease == d)]
    assert len(r) == 1, (arch, model, d, len(r))
    return r.iloc[0]

def star_cell(v, p=None, bold=False, fmt="{:.3f}", worse=False):
    """Stars for a significant test; with `worse`, a dagger instead (significantly below MSP)."""
    s = fmt.format(v)
    s = f"\\mathbf{{{s}}}" if bold else s
    sig = stars(p) if p is not None else ""
    sig = "\\dagger" if sig and worse else sig
    return f"${s}^{{{sig}}}$" if sig else f"${s}$"

def arch_table():
    long, rows = [], []
    for arch in ARCH_ORDER:
        blocks = [(d, DISEASE_NAMES[d], ARCH_SUMMARY.loc[(arch, d)]) for d in DISEASES]
        if ARCH_SHOW_MEAN:
            blocks.append(("AVERAGE", "\\textit{Mean}", ARCH_SUMMARY.loc[arch].loc[DISEASES].mean()))
        for d, name, s in blocks:
            auc  = {m: s[f"AUROC {m}"] for m in ARCH_MODELS}
            best = max(ARCH_MODELS, key=auc.get)
            gain = auc["Ensemble"] - auc["MSP"]
            t = arch_test(arch, "Ensemble", d)
            assert abs(t.gain - gain) < 1e-6, (arch, d, t.gain, gain)
            row = {"Arch": ARCH_LABELS[arch], "Disease": name,
                   "c0_auc": f"${s['C0 AUROC']:.3f}$", "c0_acc": f"${s['C0 accuracy']:.3f}$",
                   "MSP": star_cell(auc["MSP"], bold=best == "MSP"),
                   "d": star_cell(gain, t.p_adj, fmt="{:+.4f}")}
            for m in ARCH_MODELS[1:]:
                row[m] = star_cell(auc[m], arch_test(arch, m, d).p_adj, bold=best == m, worse=auc[m] < auc["MSP"])
            rows.append(row)
            long.append({"arch": arch, "disease": d, "C0 AUROC": s["C0 AUROC"], "C0 acc": s["C0 accuracy"],
                         **auc, "Ens - MSP": gain, "boot lo": t.lo, "boot hi": t.hi, "p_adj": t.p_adj})
    return pd.DataFrame(long).set_index(["arch", "disease"]), rows

arch_tab, arch_rows = arch_table()
arch_tab.round(4)


In [ ]:
tex = latex_table(
    arch_rows, ["Arch", "Disease", "c0_auc", "c0_acc", *ARCH_MODELS, "d"], group_col="Arch",
    header=["C0", "Disease", "AUROC", "Acc.", MSP_LABEL, "Attribute", "CNN", "Ours", "$\\Delta$"],
    col_fmt="ll|cc|c|ccc|r",
    pre_header=["\\multicolumn{2}{c|}{} & \\multicolumn{2}{c|}{C0} & \\multicolumn{4}{c|}{C2: AUROC for C0 correctness} "
                "& \\multicolumn{1}{c}{Ours $-$ MSP} \\\\",
                "\\cmidrule(lr){3-4} \\cmidrule(lr){5-8} \\cmidrule(l){9-9}"],
    caption=f"The final system on three C0 architectures, recipe frozen (KNN counterfactuals, $K={ARCH_CFG['k']}$). "
            "Left: C0's own performance on the held-out rows. Right: AUROC for predicting C0 correctness.",
    label="tab:c0_arch",
    notes="C0 AUROC: C0's AUROC for its disease; Acc.: its accuracy at its own Youden threshold. C2 columns: mean-fold "
          "AUROC over 5 CV folds. Attribute: MLP I+Cf+D; CNN: Dual+S; Ours: their ensemble, with "
          "the combiner chosen per disease on DenseNet121 and kept fixed. Bold: the best C2 model per row. "
          f"Stars on a model: significantly better than the adjusted MSP; $\\dagger$: significantly worse; on $\\Delta$: "
          f"significantly different from zero (its sign gives the direction). Paired patient-level bootstrap ({ARCH_CFG['n_boot']} replicates, the same "
          "patients resampled for every architecture), Holm-adjusted across the five diseases within each architecture "
          "and comparison"
          + ("; for the Mean row the test of the five-disease average" if ARCH_SHOW_MEAN else "")
          + f"; {STARS_KEY}; no star: not significant."
          + (" Mean: average over the five diseases." if ARCH_SHOW_MEAN else ""),
)
# Set the Mean row apart: a light rule across the data columns above it in each block.
tex = "\n".join("\\cmidrule(l){2-9}\n%s" % l if l.lstrip().startswith("& \\textit{Mean}") else l
                for l in tex.split("\n"))
save_table(tex, "c0_arch")


## 11. Group profiles: attribute and saliency groups, per side of C0's call

The numbers behind the group-profile figures of the two explainability notebooks, which only draw them as
dumbbells. Per disease and side of C0's call, fold 4's calls are split into groups **A, B, C** by k-means in a C2
embedding (A = lowest error rate), and each group is described on the side's key features.

* **Table 11a, attributes** (`attribute_explainability.ipynb`, step 6, figure `embedding_group_profiles`): groups in
  the attribute MLP's (I+C+D) last hidden layer, described on the side's five step-4 attributes. **I** = the group's
  mean image value, **Cf** = the mean of their counterfactuals; both in SD units of the image values over all calls
  (0 = an average image). Read from `figures/xai/<disease>/<disease>_knn_correct_cf_embedding_group_profiles.csv`.
* **Table 11b, saliency** (`saliency_explainability.ipynb`, step 6, figure `saliency_group_profiles`): groups in the
  CNN's embedding, described on the side's key regions and `Outside`. **Sup.** = log focus of the map supporting
  C0's call, **Ag.** = of the map against it; both in SD units of all calls on that side (0 = an average call).
  Read from `figures/xai/<disease>/<disease>_knn_correct_cf_saliency_group_profiles.csv`.

Only the diseases in `PROFILE_DISEASES` (cardiomegaly, effusion) are tabled. Both CSVs are written by the profile cell itself, so run each notebook per disease (`XAI_DISEASE=<disease>`) at least
down to that cell. The first row of every side gives each group's error rate and size.

In [ ]:
PROFILE_SOURCE = "knn_correct_cf"
PROFILE_SIDES  = {1: "Positive", 0: "Negative"}
PROFILE_DISEASES = ["cardiomegaly", "effusion"]   # the diseases the explainability notebooks were run for
SKIP           = "<skip>"      # a cell swallowed by the \multicolumn before it

def group_profiles(name, source=PROFILE_SOURCE):
    """The group-profile CSV `name` of every disease, stacked."""
    out = []
    for d in PROFILE_DISEASES:
        path = os.path.join(XAI_DIR, d, f"{d}_{source}_{name}.csv")
        assert os.path.exists(path), f"{path} missing: rerun the explainability notebook with XAI_DISEASE={d}"
        out.append(pd.read_csv(path))
    return pd.concat(out, ignore_index=True)

def profile_table(prof, key, forms, form_header, label_fn, caption, label, notes):
    """One block per disease and side: a row of group error rates / sizes, then one row per feature
    with the two forms of every group, side by side."""
    groups = sorted(prof["group"].unique())
    cols   = ["Disease", "call", "feat", *[f"{g}_{f}" for g in groups for f in forms]]
    rows   = []
    for d in PROFILE_DISEASES:
        for p, side in PROFILE_SIDES.items():
            t = prof[(prof["disease"] == d) & (prof["side"] == p)]
            g0 = t[t["group"] == groups[0]].sort_values("order")
            stats = {"Disease": DISEASE_NAMES[d],
                     "call": (f"\\multirow{{{len(g0) + 1}}}{{*}}{{\\shortstack[l]{{{side}\\\\"
                              f"${100 * g0['side_error_rate'].iloc[0]:.1f}\\%$ wrong}}}}"),
                     "feat": "\\textit{wrong ($n$)}"}
            for i, g in enumerate(groups):
                r = t[t["group"] == g].iloc[0]
                bar = "|" if i < len(groups) - 1 else ""
                # error rate and n on the held-out half of fold 4 (the rows k-means did not see)
                stats[f"{g}_{forms[0]}"] = (f"\\multicolumn{{2}}{{c{bar}}}{{${100 * r['error_rate_heldout']:.1f}\\%$ "
                                            f"({n_fmt(r['n_heldout'])})}}")
                stats[f"{g}_{forms[1]}"] = SKIP
            rows.append(stats)
            for _, r in g0.iterrows():
                v = t[t[key] == r[key]].set_index("group")
                rows.append({"Disease": DISEASE_NAMES[d], "call": "", "feat": label_fn(r[key]),
                             **{f"{g}_{f}": f"${v.loc[g, f]:+.2f}$" for g in groups for f in forms}})
    tex = latex_table(
        rows, cols, group_col="Disease",
        header=["Disease", "$C_0$'s call", "", *form_header * len(groups)],
        col_fmt="lll" + "|cc" * len(groups),
        pre_header=["& & & " + " & ".join(f"\\multicolumn{{2}}{{c{'|' if i < len(groups) - 1 else ''}}}{{Group {g}}}"
                                          for i, g in enumerate(groups)) + " \\\\",
                    " ".join(f"\\cmidrule(lr){{{4 + 2 * i}-{5 + 2 * i}}}" for i in range(len(groups)))],
        caption=caption, label=label, notes=notes)
    tex = tex.replace(" & " + SKIP, "")
    # A lighter rule between the two sides of one disease, and a smaller font for nine columns.
    tex = re.sub(r"\n & (\\multirow\{\d+\}\{\*\}\{\\shortstack\[l\]\{Negative)",
                 lambda m: f"\n\\cmidrule(l){{2-{len(cols)}}}\n & {m.group(1)}", tex)
    tex = tex.replace("\\small", "\\footnotesize", 1)
    return tex

In [ ]:
attr_prof = group_profiles("embedding_group_profiles")
tex = profile_table(
    attr_prof, "attribute", ["image", "CF"], ["I", "Cf"], attr_label,
    caption="Attribute profiles of the groups in the attribute model's embedding, per disease and side of $C_0$'s "
            "call: group means of the side's five most error-related attributes, for the image (I) and its "
            "counterfactual (Cf), in standard deviations (KNN counterfactuals, nearest counterfactual, fold 4).",
    label="tab:attr_group_profiles",
    notes="Groups: k-means ($k=3$) on the last hidden layer of the attribute MLP (I+Cf+D), per side of $C_0$'s call; "
          "A has the lowest error rate. Attributes: the side's five non-redundant attributes most correlated with "
          "correctness (Table~\\ref{tab:top_attributes}). Values: group mean, z-scored with the mean and standard "
          "deviation of the image values over all calls (0 = an average image); the counterfactual is z-scored "
          "with the same image statistics, so I $-$ Cf is the shift between an image and its counterfactual. "
          "First row of each side: the group's $C_0$ error rate and number of calls on the held-out half of fold 4 "
          "(the patient-disjoint half k-means was not fitted on); the profile values are means over the whole group. "
          "Side label: error rate over all of that side's fold-4 calls.",
)
save_table(tex, "attr_group_profiles")
attr_prof.assign(attribute=attr_prof["attribute"].map(attr_label)).pivot_table(
    index=["disease", "side", "order", "attribute"], columns="group", values=["image", "CF"]
)[["image", "CF"]].sort_index(level=[0, 1, 2], ascending=[True, False, True]).round(2)

In [ ]:
sal_prof = group_profiles("saliency_group_profiles")
tex = profile_table(
    sal_prof, "region", ["support", "against"], ["Sup.", "Ag."], region_label,
    caption="Saliency profiles of the groups in the CNN's embedding, per disease and side of $C_0$'s call: group "
            "means of the log focus of the Grad-CAM supporting $C_0$'s call (Sup.) and of the one against it (Ag.), "
            "on the side's key regions, in standard deviations (KNN counterfactuals, fold 4).",
    label="tab:sal_group_profiles",
    notes="Groups: k-means ($k=3$) on the embedding of the CNN, per side of $C_0$'s call; A has the lowest error "
          "rate. Regions: the side's four non-redundant regions whose focus correlates most with correctness, "
          "and Outside (the image outside the body). Focus: share of the map in the region divided by the "
          "region's share of the image, as $\\log_{10}$; values are group means, z-scored over all calls on that "
          "side (0 = an average call). First row of each side: the group's $C_0$ error rate and number of calls on the "
          "held-out half of fold 4 (the patient-disjoint half k-means was not fitted on); the profile values are "
          "means over the whole group. Side label: error rate over all of that side's fold-4 calls.",
)
save_table(tex, "sal_group_profiles")
sal_prof.assign(region=sal_prof["region"].map(region_label)).pivot_table(
    index=["disease", "side", "order", "region"], columns="group", values=["support", "against"]
)[["support", "against"]].sort_index(level=[0, 1, 2], ascending=[True, False, True]).round(2)